# K-Means Clustering of European Regions by Population Structure

**Dataset:** NUTS3 demographic data — ~1500 regions across Europe  
**Method:** K-Means with K = 5 to 30, evaluated by 5 quality metrics  
**Visualization:** Sankey flow diagrams, geographic maps, cluster profiles  

**Learning objectives:**
1. Understand how K-Means partitions data and how to choose K
2. Interpret clustering quality metrics (Silhouette, SSE, Variance Explained, etc.)
3. Use dimensionality reduction (MDS/UMAP) to assign visually consistent colors
4. Read Sankey diagrams to understand how clusters evolve with changing parameters
5. Interpret cluster profiles using Z-scores and radar charts
6. Connect abstract clusters to real-world geography

---

# 🎯 K-Means Clustering of European NUTS3 Regions by Population Structure

This notebook demonstrates **iterative K-Means clustering** applied to demographic data
for ~1500 European NUTS3 regions. We explore how population age/gender profiles
define natural groupings of regions, and how the number of clusters (K) affects
the result.

## Workflow Overview

| Step | What | Why |
|------|-------|-----|
| 1 | Load demographic data | Raw material for clustering |
| 2 | Load geographic boundaries | To visualize clusters on a map |
| 3 | Run K-Means for K=5..30 | Explore the effect of K on cluster quality |
| 4 | Evaluate quality metrics | Find the "best" K objectively |
| 5 | Assign consistent colors | Enable visual comparison across K values |
| 6 | 1D Sankey embedding | Track how clusters split/merge as K increases |
| 7 | Cluster profiles | Understand *what* each cluster represents |
| 8 | Interactive map | See *where* the clusters are geographically |

## Requirements

pip install pandas numpy scikit-learn geopandas matplotlib plotly ipywidgets shapely

Optional: `pip install umap-learn` for better dimensionality reduction.

---

### Cell 1: Import Libraries

We load all necessary packages upfront:
- **pandas / numpy** — data handling
- **geopandas** — spatial data (shapefiles)
- **matplotlib / plotly** — visualization
- **scikit-learn** — K-Means, scaling, quality metrics
- **ipywidgets** — interactive UI controls in the notebook

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import warnings, os

warnings.filterwarnings('ignore')
%matplotlib inline

DATA_DIR = os.path.join("data", "EU-NUTS3-2026")
print("Data directory:", os.path.abspath(DATA_DIR))
print("Files:", sorted(os.listdir(DATA_DIR)))

### Cell 2: Load and Prepare the Demographic Data

**What this cell does:**
1. Loads the CSV file with population data for all NUTS3 regions across multiple years
2. Filters to the **latest available year** (so we cluster a single time snapshot)
3. Selects the **feature columns** — the percentages that describe each region's population structure
4. Drops rows with missing values (regions where data is incomplete)

**Feature columns used for clustering:**

| Feature | Description |
|---------|-------------|
| `pct_female` | % female population |
| `pct_0_14` | % aged 0–14 (children) |
| `pct_15_24` | % aged 15–24 (youth) |
| `pct_25_44` | % aged 25–44 (young adults) |
| `pct_45_64` | % aged 45–64 (middle-aged) |
| `pct_65_plus` | % aged 65+ (elderly) |

> **Note:** `pct_male` is excluded because it always equals `100 - pct_female`,
> which would introduce perfect multicollinearity — a redundant dimension
> that adds no information but can distort distance calculations.

In [ ]:
# ── Load population CSV ──

pop_df = pd.read_csv(os.path.join(DATA_DIR, "nuts3_population.csv"))
print(f"Loaded {len(pop_df)} rows, {len(pop_df.columns)} columns")
print(f"Countries: {sorted(pop_df['ID_NUTS0'].unique())}")

# ── year = last year of data collection (one record per region) ──

year_counts = pop_df['year'].value_counts().sort_index()
print(f"\nRegions by data collection year:")
for yr, cnt in year_counts.items():
    print(f"  {yr}: {cnt} regions")

YEAR_MIN = int(pop_df['year'].min())
YEAR_MAX = int(pop_df['year'].max())

# ── Use ALL rows (no filtering) ──

df = pop_df.copy()
print(f"\nTotal regions: {len(df)} (years {YEAR_MIN}–{YEAR_MAX})")

# ── Drop rows with missing features ──

FEATURE_COLS = [#'pct_male', — excluded: sums to 100% with pct_female
                'pct_female',
                'pct_0_14', 'pct_15_24', 'pct_25_44', 'pct_45_64', 'pct_65_plus']
before = len(df)
df = df.dropna(subset=FEATURE_COLS).reset_index(drop=True)
print(f"After dropping NaN features: {len(df)} rows ({before - len(df)} removed)")
display(df[FEATURE_COLS].describe().round(2))

# ── Show countries with older data (for awareness) ──

country_years = (df.groupby('ID_NUTS0')['year']
                 .agg(['min', 'max', 'count'])
                 .sort_values('max'))
older = country_years[country_years['max'] < YEAR_MAX]
if len(older) > 0:
    print(f"\n⚠ Countries with data collected before {YEAR_MAX}:")
    for country, row in older.iterrows():
        print(f"  {country}: {int(row['count'])} regions, "
              f"collected {int(row['min'])}–{int(row['max'])}")
else:
    print(f"\n✅ All countries have data collected in {YEAR_MAX}")

display(df.head(8))

### Cell 3: Load Geographic Boundaries (Shapefiles)

To display clustering results on a **map**, we need the polygon boundaries
of each NUTS3 region. These come from ESRI Shapefiles (`.shp` + companion files).

**What this cell does:**
1. Loads the **NUTS3 shapefile** (~1500 region polygons)
2. **Auto-detects the join column** - the column in the shapefile that matches

   our `ID_NUTS3` codes (e.g., `NUTS_ID`). Different shapefile sources use
   different column names, so we try several candidates and pick the one
   with the most matches.
3. Loads the **NUTS0 shapefile** (country-level borders) - used later

   to draw thick country outlines on top of the cluster map.

> **NUTS** = Nomenclature of Territorial Units for Statistics. It's the EU's
> hierarchical system for dividing territory: NUTS0 = countries, NUTS1 = major
> regions, NUTS2 = medium regions, NUTS3 = small regions (~1500 across Europe).

In [ ]:
# ── Load NUTS3 shapefile ──

nuts3_shp = gpd.read_file(os.path.join(DATA_DIR, "NUTS3.shp"))
print(f"Shapefile: {len(nuts3_shp)} geometries, CRS={nuts3_shp.crs}")
print(f"Columns: {list(nuts3_shp.columns)}")

# ── Identify the join column (varies by source) ──

join_candidates = ['NUTS_ID', 'NUTS3_ID', 'NUTS_CODE', 'id', 'ID', 'CNTR_ID']
SHP_ID_COL = None
for c in nuts3_shp.columns:
    if c in join_candidates or 'NUTS' in c.upper():
        # Check overlap with our data
        overlap = set(nuts3_shp[c].astype(str)) & set(df['ID_NUTS3'].astype(str))
        if len(overlap) > 10:
            SHP_ID_COL = c
            print(f"Join column: '{c}' ({len(overlap)} matches)")
            break

if SHP_ID_COL is None:
    # Fallback: show first 5 rows so user can identify
    print("⚠ Could not auto-detect join column. Shapefile sample:")
    display(nuts3_shp.head())
    SHP_ID_COL = input("Enter shapefile column name for NUTS3 ID: ").strip()

# ── Also load NUTS0 for country borders ──

nuts0_shp = gpd.read_file(os.path.join(DATA_DIR, "NUTS0.shp"))
print(f"NUTS0 borders: {len(nuts0_shp)} countries")

### Cell 4: Run K-Means Clustering for K = 5 to 30

This is the main computation cell. We run K-Means **26 times** with different
values of K (number of clusters), each time recording five quality metrics.

**Key steps:**

1. **Z-score normalization** (`StandardScaler`): Each feature is transformed to

   have mean=0 and std=1. This prevents features with larger numeric ranges
   (e.g., `pct_25_44 ≈ 25%`) from dominating over features with smaller ranges
   (e.g., `pct_female ≈ 50%` with very low variance).

2. **Total Sum of Squares (TSS)**: The total variance in the data before any

   clustering — needed to compute Variance Explained.

3. **K-Means loop**: For each K, we fit the model and compute:

| Metric | Formula / Idea | Better when... |
|--------|---------------|----------------|
| **SSE** (Inertia) | Sum of squared distances to nearest centroid | **Lower** (tighter clusters) |
| **Variance Explained** | `(1 − SSE/TSS) × 100%` | **Higher** (more structure captured) |
| **Silhouette Score** | How similar points are to own cluster vs. nearest other | **Higher** (range −1 to +1) |
| **Calinski-Harabasz** | Ratio of between-cluster to within-cluster variance | **Higher** |
| **Davies-Bouldin** | Average similarity between each cluster and its most similar | **Lower** (less overlap) |

> **Why multiple metrics?** No single metric tells the whole story. SSE always
> decreases with more clusters (overfitting). Silhouette captures separation.
> Calinski-Harabasz and Davies-Bouldin measure different aspects of cluster geometry.
> The "best" K is where **multiple metrics agree** — often visible as an "elbow" or peak.

In [ ]:
# ── Cell 4 — Run K-Means Iteratively with Progress Bar ───────

# ── Parameters ──

K_MIN, K_MAX, SEED = 5, 30, 1
N_ITERATIONS = K_MAX - K_MIN + 1

# ── Normalize features ──

scaler = StandardScaler()
X = scaler.fit_transform(df[FEATURE_COLS].values)
print(f"Feature matrix: {X.shape[0]} regions × {X.shape[1]} features (Z-scored)")

# ── Total Sum of Squares (for Variance Explained) ──

centroid_global = X.mean(axis=0)
TSS = float(np.sum((X - centroid_global) ** 2))

# ── Progress widgets ──

progress_bar_km = widgets.IntProgress(
    value=0, min=0, max=N_ITERATIONS,
    bar_style='info',
    layout=widgets.Layout(width='600px'))
progress_label_km = widgets.HTML(value='')
progress_box = widgets.HBox([progress_bar_km, progress_label_km])
display(progress_box)

# ── Run iterations ──

import time
t0 = time.time()

results = []
cluster_assignments = {}   # key=K → array of labels

for i, k in enumerate(range(K_MIN, K_MAX + 1)):
    t_iter = time.time()

    progress_bar_km.value = i
    progress_label_km.value = (
        f"<span style='font-size:12px; color:#555;'>"
        f"⏳ K={k} ({i+1}/{N_ITERATIONS})...</span>")

    km = KMeans(n_clusters=k, random_state=SEED, n_init=10, max_iter=300)
    labels = km.fit_predict(X)

    sse = float(km.inertia_)
    var_explained = (1.0 - sse / TSS) * 100.0 if TSS > 0 else 0.0
    sil = silhouette_score(X, labels) if k > 1 else 0.0
    ch = calinski_harabasz_score(X, labels) if k > 1 else 0.0
    db = davies_bouldin_score(X, labels) if k > 1 else 0.0

    results.append({
        'K': k,
        'SSE': sse,
        'Variance_Explained': var_explained,
        'Silhouette': sil,
        'Calinski_Harabasz': ch,
        'Davies_Bouldin': db
    })
    cluster_assignments[k] = labels

    dt = time.time() - t_iter
    print(f"  K={k:2d}  SSE={sse:8.1f}  VarExpl={var_explained:5.1f}%  "
          f"Silh={sil:.4f}  CH={ch:.1f}  DB={db:.4f}  ({dt:.2f}s)")

# ── Finish ──

progress_bar_km.value = N_ITERATIONS
progress_bar_km.bar_style = 'success'
elapsed = time.time() - t0
progress_label_km.value = (
    f"<span style='font-size:12px; color:#2E7D32; font-weight:bold;'>"
    f"✅ Done — {N_ITERATIONS} iterations in {elapsed:.1f}s</span>")

metrics_df = pd.DataFrame(results)
print(f"\nDone: {len(metrics_df)} iterations in {elapsed:.1f}s")
display(metrics_df.style
        .format(precision=4)
        .background_gradient(subset=['Silhouette'], cmap='Greens')
        .background_gradient(subset=['Davies_Bouldin'], cmap='Reds_r')
        .background_gradient(subset=['Variance_Explained'], cmap='Blues'))

### Cell 5: Visualize Quality Metrics Across All K Values

Now we plot each metric as a function of K to visually identify the **optimal
number of clusters**.

**What to look for:**

| Chart | Pattern for optimal K |
|-------|----------------------|
| SSE | **Elbow** — the point where the curve bends and flattening begins |
| Variance Explained | **Elbow** — diminishing returns start |
| Silhouette | **Peak** — highest value = best separation |
| Calinski-Harabasz | **Peak** — highest ratio of between/within variance |
| Davies-Bouldin | **Valley** — lowest overlap between clusters |

The cell first tries **Plotly** (interactive hover tooltips) and falls back to
**matplotlib** if Plotly isn't configured for the notebook environment.

> **Practical tip:** The "elbow" is often subjective. In real-world analyses,
> you might pick 2–3 candidate K values and examine the actual clusters
> (on a map, via profiles) before making a final decision.

In [ ]:
# ── Cell 5 — Metrics Charts ──────────────────────────────────

# Try to configure Plotly renderer for this environment

import plotly.io as pio

# Uncomment ONE of these if charts still don't appear:

pio.renderers.default = 'notebook'     # classic Jupyter

# pio.renderers.default = 'iframe'       # works in most environments

# pio.renderers.default = 'browser'      # opens in external browser

METRIC_INFO = {
    'SSE':                {'color': '#1565C0', 'lower_better': True,
                           'title': 'Sum of Squared Errors'},
    'Variance_Explained': {'color': '#2E7D32', 'lower_better': False,
                           'title': 'Variance Explained (%)'},
    'Silhouette':         {'color': '#E65100', 'lower_better': False,
                           'title': 'Silhouette Score'},
    'Calinski_Harabasz':  {'color': '#6A1B9A', 'lower_better': False,
                           'title': 'Calinski-Harabasz Index'},
    'Davies_Bouldin':     {'color': '#C62828', 'lower_better': True,
                           'title': 'Davies-Bouldin Index'},
}

fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[v['title'] for v in METRIC_INFO.values()] + [''],
    vertical_spacing=0.10, horizontal_spacing=0.08
)

for i, (metric, info) in enumerate(METRIC_INFO.items()):
    row, col = (i // 2) + 1, (i % 2) + 1
    fig.add_trace(go.Scatter(
        x=metrics_df['K'], y=metrics_df[metric],
        mode='lines+markers',
        marker=dict(size=8, color=info['color']),
        line=dict(color=info['color'], width=2),
        name=metric,
        hovertemplate=f"<b>{metric}</b><br>K=%{{x}}<br>Value=%{{y:.4f}}<extra></extra>"
    ), row=row, col=col)
    fig.update_xaxes(title_text="K", row=row, col=col)
    fig.update_yaxes(title_text=metric, row=row, col=col)

fig.update_layout(height=800, showlegend=False,
                  title_text="K-Means Quality Metrics (seed=1, K=5..30)")

# ── Fallback: if Plotly doesn't render, use matplotlib ──

try:
    fig.show()
except Exception as e:
    print(f"Plotly rendering failed ({e}), falling back to matplotlib...")
    
    fig_mpl, axes = plt.subplots(3, 2, figsize=(14, 10))
    axes = axes.flatten()
    
    for i, (metric, info) in enumerate(METRIC_INFO.items()):
        ax = axes[i]
        ax.plot(metrics_df['K'], metrics_df[metric],
                'o-', color=info['color'], markersize=6, linewidth=2)
        ax.set_xlabel('K')
        ax.set_ylabel(metric)
        ax.set_title(info['title'], fontsize=11)
        ax.grid(True, alpha=0.3)
        
        # Mark best value
        if info['lower_better']:
            best_idx = metrics_df[metric].idxmin()
        else:
            best_idx = metrics_df[metric].idxmax()
        best_k = metrics_df.loc[best_idx, 'K']
        best_v = metrics_df.loc[best_idx, metric]
        ax.axvline(x=best_k, color=info['color'], linestyle='--', alpha=0.4)
        ax.annotate(f'best K={best_k}', xy=(best_k, best_v),
                    fontsize=8, color=info['color'],
                    textcoords="offset points", xytext=(10, 5))
    
    # Hide unused subplot
    axes[-1].set_visible(False)
    
    fig_mpl.suptitle(f"K-Means Quality Metrics (seed={SEED}, K={K_MIN}..{K_MAX})",
                     fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

###### Cell 6: Assign Consistent Colors via Common Centroid Projection

**The problem:** When we change K (e.g., from 10 to 11), cluster numbering is
arbitrary — "Cluster 3 at K=10" has no relation to "Cluster 3 at K=11". If we
assign colors by cluster number, colors jump randomly between K values, making
visual comparison impossible.

**The solution:** Project **all centroids from all K values** into a shared 2D
space, then assign colors based on their **position** in that space.

#### How it works (4 steps):

1. **Pool** all centroids: K=5 has 5 centroids, K=6 has 6, ..., K=30 has 30.

   Total: 5+6+...+30 = 455 centroid vectors in the original feature space.

2. **Project** all 455 centroids into **one common 2D plane** using MDS

   (Multidimensional Scaling) or UMAP. This preserves distances: centroids
   that are similar in the 6D feature space land nearby in 2D.

3. **Map position → color** using polar coordinates from the center:
   - **Angle** from center → **Hue** (position on the color wheel)
   - **Distance** from center → **Saturation** (far = vivid, near = pastel)

4. **Split** colors back to per-K dictionaries.

**Key benefit:** A cluster at K=10 that represents "young, high-fertility regions"
will land near the same 2D position as the corresponding cluster at K=11 — and
therefore receive a **similar color**. This makes it visually obvious how clusters
evolve as K increases.

The visualization at the end shows two scatter plots:
- **Left**: all centroids colored by their K value
- **Right**: all centroids colored by their assigned HLS color

In [ ]:
# ── Cell 6 — Common centroid projection & color assignment ───

from sklearn.manifold import MDS
from colorsys import hls_to_rgb
from scipy.spatial.distance import pdist, squareform

try:
    from umap import UMAP
    HAS_UMAP = True
    print("✅ UMAP available")
except ImportError:
    HAS_UMAP = False
    print("ℹ️  UMAP not installed — using MDS (pip install umap-learn)")

# ═══════════════════════════════════════════════════════════════

#  Step 1: Collect ALL centroids from ALL K values into one pool

# ═══════════════════════════════════════════════════════════════

all_centroids = []       # list of (k, cluster_idx, centroid_vector)
centroid_index = []      # parallel list of (k, cluster_idx) tuples

for k, labels in cluster_assignments.items():
    for c in range(k):
        mask = labels == c
        if mask.sum() > 0:
            centroid = X[mask].mean(axis=0)
        else:
            centroid = np.zeros(X.shape[1])
        all_centroids.append(centroid)
        centroid_index.append((k, c))

all_centroids = np.array(all_centroids)
print(f"Total centroids pooled: {len(all_centroids)} "
      f"(from K={K_MIN}..{K_MAX})")

# ═══════════════════════════════════════════════════════════════

#  Step 2: Project ALL centroids into a SINGLE 2D space

# ═══════════════════════════════════════════════════════════════

n_pts = all_centroids.shape[0]

if HAS_UMAP and n_pts >= 15:
    n_neighbors = min(n_pts - 1, 15)
    reducer = UMAP(n_components=2, n_neighbors=n_neighbors,
                   min_dist=0.3, random_state=42)
    all_coords_2d = reducer.fit_transform(all_centroids)
    EMBED_METHOD = "UMAP"
else:
    mds = MDS(n_components=2, random_state=42,
              dissimilarity='euclidean', normalized_stress='auto',
              n_init=8, max_iter=500)
    all_coords_2d = mds.fit_transform(all_centroids)
    EMBED_METHOD = "MDS"

print(f"Projection method: {EMBED_METHOD}")

# Normalize to [0, 1] globally

for dim in range(2):
    mn, mx = all_coords_2d[:, dim].min(), all_coords_2d[:, dim].max()
    rng = mx - mn
    if rng > 1e-10:
        all_coords_2d[:, dim] = (all_coords_2d[:, dim] - mn) / rng
    else:
        all_coords_2d[:, dim] = 0.5

# ═══════════════════════════════════════════════════════════════

#  Step 3: Assign colors based on global 2D position

# ═══════════════════════════════════════════════════════════════

def coords_to_colors(coords_2d):
    """
    Map normalized 2D coordinates to distinguishable colors.
    angle from center → Hue, distance from center → Saturation.
    """
    cx, cy = coords_2d[:, 0].mean(), coords_2d[:, 1].mean()
    dx = coords_2d[:, 0] - cx
    dy = coords_2d[:, 1] - cy
    angles = np.arctan2(dy, dx)
    hues = (angles + np.pi) / (2 * np.pi)

    dists = np.sqrt(dx**2 + dy**2)
    max_dist = dists.max() if dists.max() > 1e-10 else 1.0
    dists_norm = dists / max_dist

    colors = []
    for h, d in zip(hues, dists_norm):
        sat = 0.45 + 0.50 * d
        lit = 0.38 + 0.22 * (1 - d)
        r, g, b = hls_to_rgb(h, lit, sat)
        colors.append(f'#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}')
    return colors

# Compute colors for ALL centroids at once (global consistency)

all_colors = coords_to_colors(all_coords_2d)

# ═══════════════════════════════════════════════════════════════

#  Step 4: Split back into per-K dictionaries

# ═══════════════════════════════════════════════════════════════

centroid_data = {}  # key=K → {'colors': [...], 'centroids': array, 'coords_2d': array}

for i, (k, c_idx) in enumerate(centroid_index):
    if k not in centroid_data:
        centroid_data[k] = {
            'colors': [None] * k,
            'centroids': np.zeros((k, X.shape[1])),
            'coords_2d': np.zeros((k, 2))
        }
    centroid_data[k]['colors'][c_idx] = all_colors[i]
    centroid_data[k]['centroids'][c_idx] = all_centroids[i]
    centroid_data[k]['coords_2d'][c_idx] = all_coords_2d[i]

# ═══════════════════════════════════════════════════════════════

#  Step 5: Visualize — show global embedding colored by K

# ═══════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# ── Left: ALL centroids in the common 2D space, colored by K ──

ax = axes[0]
ks_arr = np.array([ci[0] for ci in centroid_index])
unique_ks = sorted(set(ks_arr))
k_cmap = plt.cm.get_cmap('viridis', len(unique_ks))

for j, kv in enumerate(unique_ks):
    mask = ks_arr == kv
    ax.scatter(all_coords_2d[mask, 0], all_coords_2d[mask, 1],
               c=[k_cmap(j)], s=30, alpha=0.6, label=f'K={kv}')

ax.set_title(f'All centroids in common {EMBED_METHOD} space\n(colored by K)',
             fontsize=11, fontweight='bold')
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 1.05)
ax.set_aspect('equal')
ax.grid(True, alpha=0.2)
ax.legend(fontsize=6, ncol=3, loc='upper right', framealpha=0.8)

# ── Right: ALL centroids colored by assigned HLS color ──

ax = axes[1]
for i in range(len(all_centroids)):
    ax.scatter(all_coords_2d[i, 0], all_coords_2d[i, 1],
               c=all_colors[i], s=40, edgecolors='black',
               linewidths=0.3, alpha=0.8)

ax.set_title(f'All centroids — assigned colors\n(similar position = similar color)',
             fontsize=11, fontweight='bold')
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 1.05)
ax.set_aspect('equal')
ax.grid(True, alpha=0.2)

plt.suptitle(f'Common {EMBED_METHOD} projection: {len(all_centroids)} centroids '
             f'from K={K_MIN}..{K_MAX}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Sample output ──

sample_k = min(10, K_MAX)
cd = centroid_data[sample_k]
print(f"\nSample: K={sample_k}")
for c_idx in range(sample_k):
    n_pts = (cluster_assignments[sample_k] == c_idx).sum()
    xy = cd['coords_2d'][c_idx]
    print(f"  Cluster {c_idx:2d}: color={cd['colors'][c_idx]}  "
          f"pos=({xy[0]:.2f}, {xy[1]:.2f})  n={n_pts}")

### Cell 7 (1D Embedding): Track Cluster Evolution with Sankey Flow

This is arguably the most informative visualization in the notebook. It answers:
**"What happens to the data points as I increase K?"**

#### Layout concept

Each **vertical column** represents one K value. Within each column, clusters are
drawn as **stacked bars** (height ∝ cluster size). Between adjacent columns,
**Sankey ribbons** show how many data points flow from each cluster at K to each
cluster at K+1.

#### How it's built:

| Step | Detail |
|------|--------|
| **1D Embedding** | All 455 centroids are projected to **1D** (MDS, PCA, or UMAP). This determines the **vertical ordering** of clusters within each column. |
| **Bar ordering** | Clusters sorted by their 1D coordinate — similar clusters appear at the same vertical position across columns. |
| **Gap sizing** | Gaps between bars are proportional to 1D distance — visually separating dissimilar clusters. |
| **Sankey ribbons** | For adjacent K values, we count how many data points belong to cluster `i` at K and cluster `j` at K+1. Ribbon width ∝ overlap count. |
| **Ribbon color** | Blend of source and target cluster colors. |
| **Crossing minimization** | Ribbons are sorted by source+target 1D position to reduce visual clutter. |

#### What to look for:
- **Clean parallel ribbons** → Cluster structure is stable; increasing K just splits one cluster
- **Many crossing ribbons** → Cluster structure is unstable; the partition reorganizes substantially
- **One bar splitting into two** → A cluster at K naturally divides at K+1 (good sign)
- **Many small ribbons** → Points scatter to many clusters (possibly overfitting)

#### Interactive controls:
- **K range** — zoom into a subset of iterations
- **1D method** — MDS (default), PCA, or UMAP for the vertical ordering
- **Show Sankey** — toggle ribbons on/off
- **Ribbon α** — transparency of the flow ribbons

In [ ]:
# ── Cell 7 — 1D Embedding: all iterations, Sankey connections ─

import matplotlib.path as mpath
import matplotlib.patches as mpatches

# ═══════════════════════════════════════════════════════════════

#  Step 1: 1D embedding of ALL centroids in a common space

# ═══════════════════════════════════════════════════════════════

print("Computing 1D embedding of all centroids...")

from sklearn.decomposition import PCA

EMBED_1D_METHODS = ['MDS', 'PCA']
if HAS_UMAP:
    EMBED_1D_METHODS.insert(0, 'UMAP')

def compute_1d_embedding(method='MDS'):
    if method == 'UMAP' and HAS_UMAP:
        n_neighbors = min(len(all_centroids) - 1, 15)
        reducer = UMAP(n_components=1, n_neighbors=n_neighbors,
                       min_dist=0.1, random_state=42, metric='euclidean')
        coords = reducer.fit_transform(all_centroids).flatten()
    elif method == 'PCA':
        pca = PCA(n_components=1, random_state=42)
        coords = pca.fit_transform(all_centroids).flatten()
    else:
        mds = MDS(n_components=1, random_state=42,
                  dissimilarity='euclidean', normalized_stress='auto',
                  n_init=10, max_iter=500)
        coords = mds.fit_transform(all_centroids).flatten()

    mn, mx = coords.min(), coords.max()
    if (mx - mn) > 1e-10:
        coords = (coords - mn) / (mx - mn)
    else:
        coords[:] = 0.5
    return coords

embed_1d_cache = {}
for method in EMBED_1D_METHODS:
    print(f"  {method}...", end=" ")
    embed_1d_cache[method] = compute_1d_embedding(method)
    print("done")

def get_coords_1d_per_k(method):
    raw = embed_1d_cache[method]
    result = {}
    for i, (k, c_idx) in enumerate(centroid_index):
        if k not in result:
            result[k] = {}
        result[k][c_idx] = raw[i]
    return result

# ═══════════════════════════════════════════════════════════════

#  Step 2: Compute overlap between ALL adjacent iterations

# ═══════════════════════════════════════════════════════════════

sorted_ks = sorted(cluster_assignments.keys())

sankey_flows = {}
for idx in range(len(sorted_ks) - 1):
    k_l, k_r = sorted_ks[idx], sorted_ks[idx + 1]
    lab_l = cluster_assignments[k_l]
    lab_r = cluster_assignments[k_r]
    flows = []
    for c_l in range(k_l):
        mask_l = (lab_l == c_l)
        for c_r in range(k_r):
            overlap = int(np.sum(mask_l & (lab_r == c_r)))
            if overlap > 0:
                flows.append((c_l, c_r, overlap))
    sankey_flows[(k_l, k_r)] = flows

print(f"\nSankey flows computed for {len(sankey_flows)} adjacent pairs")
print(f"Total flow entries: {sum(len(v) for v in sankey_flows.values())}")

# ═══════════════════════════════════════════════════════════════

#  Step 3: Drawing function

# ═══════════════════════════════════════════════════════════════

def plot_1d_sankey(k_range, embed_method='MDS', show_sankey=True,
                   show_labels=True, sankey_alpha=0.20):
    k_start, k_end = k_range
    ks = [k for k in sorted_ks if k_start <= k <= k_end]
    n_iters = len(ks)

    if n_iters < 1:
        print("Select at least 1 iteration")
        return

    coords_1d = get_coords_1d_per_k(embed_method)

    BAR_WIDTH = 0.6
    GAP_FRAC = 0.012
    TOTAL_HEIGHT = 1.0
    N_TOTAL = X.shape[0]

    fig_w = max(10, n_iters * 0.9 + 3)
    fig, ax = plt.subplots(figsize=(fig_w, 9))

    # ── Bars ──
    bar_pos = {}

    for iter_idx, k in enumerate(ks):
        labels = cluster_assignments[k]
        colors = centroid_data[k]['colors']
        x = iter_idx

        order = sorted(range(k),
                       key=lambda c: coords_1d[k].get(c, 0))

        sizes = {c: int((labels == c).sum()) for c in range(k)}
        total = sum(sizes.values())

        sorted_coords = [coords_1d[k][c] for c in order]
        if len(sorted_coords) > 1:
            raw_gaps = [sorted_coords[i+1] - sorted_coords[i]
                        for i in range(len(sorted_coords) - 1)]
            gap_sum = sum(raw_gaps)
            total_gap = (k - 1) * GAP_FRAC * TOTAL_HEIGHT
            if gap_sum > 1e-10:
                gaps = [(g / gap_sum) * total_gap for g in raw_gaps]
            else:
                gaps = [total_gap / (k - 1)] * (k - 1)
        else:
            gaps = []
            total_gap = 0.0

        avail = TOTAL_HEIGHT - sum(gaps)

        y = 0.0
        for oi, c in enumerate(order):
            h = (sizes[c] / total) * avail if total > 0 else 0

            rect = plt.Rectangle(
                (x - BAR_WIDTH / 2, y), BAR_WIDTH, h,
                facecolor=colors[c], edgecolor='black',
                linewidth=0.5, alpha=0.92, zorder=3)
            ax.add_patch(rect)

            if show_labels and h > 0.018:
                lbl = f'{c} ({sizes[c]})' if h > 0.045 else f'{c}'
                fs = 6.5 if h > 0.045 else 5.5
                bg = colors[c]
                r, g, b = int(bg[1:3],16)/255, int(bg[3:5],16)/255, int(bg[5:7],16)/255
                lum = 0.299*r + 0.587*g + 0.114*b
                tc = 'white' if lum < 0.45 else 'black'
                ax.text(x, y + h / 2, lbl, ha='center', va='center',
                        fontsize=fs, fontweight='bold', color=tc, zorder=4)

            bar_pos[(k, c)] = (y, y + h, x)
            if oi < len(gaps):
                y += h + gaps[oi]
            else:
                y += h

    # ── Sankey ribbons ──
    if show_sankey and n_iters >= 2:
        left_used = {key: 0.0 for key in bar_pos}
        right_used = {key: 0.0 for key in bar_pos}

        for iter_idx in range(len(ks) - 1):
            k_l, k_r = ks[iter_idx], ks[iter_idx + 1]
            key = (k_l, k_r)
            if key not in sankey_flows:
                continue

            flows = sorted(sankey_flows[key],
                           key=lambda f: (coords_1d[k_l].get(f[0], 0),
                                          coords_1d[k_r].get(f[1], 0)))

            for c_l, c_r, count in flows:
                if (k_l, c_l) not in bar_pos or (k_r, c_r) not in bar_pos:
                    continue

                yb_l, yt_l, x_l = bar_pos[(k_l, c_l)]
                yb_r, yt_r, x_r = bar_pos[(k_r, c_r)]
                h_l = yt_l - yb_l
                h_r = yt_r - yb_r

                size_l = (cluster_assignments[k_l] == c_l).sum()
                size_r = (cluster_assignments[k_r] == c_r).sum()

                rh_l = (count / size_l) * h_l if size_l > 0 else 0
                rh_r = (count / size_r) * h_r if size_r > 0 else 0

                y0_l = yb_l + left_used.get((k_l, c_l), 0)
                y1_l = y0_l + rh_l
                y0_r = yb_r + right_used.get((k_r, c_r), 0)
                y1_r = y0_r + rh_r

                left_used[(k_l, c_l)] = left_used.get((k_l, c_l), 0) + rh_l
                right_used[(k_r, c_r)] = right_used.get((k_r, c_r), 0) + rh_r

                xl = x_l + BAR_WIDTH / 2
                xr = x_r - BAR_WIDTH / 2
                xc1 = xl + (xr - xl) * 0.4
                xc2 = xl + (xr - xl) * 0.6

                verts = [
                    (xl, y0_l),
                    (xc1, y0_l), (xc2, y0_r), (xr, y0_r),
                    (xr, y1_r),
                    (xc2, y1_r), (xc1, y1_l), (xl, y1_l),
                    (xl, y0_l),
                ]
                codes = [
                    mpath.Path.MOVETO,
                    mpath.Path.CURVE4, mpath.Path.CURVE4, mpath.Path.CURVE4,
                    mpath.Path.LINETO,
                    mpath.Path.CURVE4, mpath.Path.CURVE4, mpath.Path.CURVE4,
                    mpath.Path.CLOSEPOLY,
                ]

                col_l = centroid_data[k_l]['colors'][c_l]
                col_r = centroid_data[k_r]['colors'][c_r]
                rl, gl, bl = int(col_l[1:3],16), int(col_l[3:5],16), int(col_l[5:7],16)
                rr, gr, br = int(col_r[1:3],16), int(col_r[3:5],16), int(col_r[5:7],16)
                blend = f'#{(rl+rr)//2:02x}{(gl+gr)//2:02x}{(bl+br)//2:02x}'

                path = mpath.Path(verts, codes)
                patch = mpatches.PathPatch(
                    path, facecolor=blend,
                    alpha=sankey_alpha, edgecolor='none', zorder=1)
                ax.add_patch(patch)

    # ── Formatting ──
    ax.set_xlim(-0.8, n_iters - 0.2)
    ax.set_ylim(-0.02, TOTAL_HEIGHT + 0.02)
    ax.set_xticks(range(n_iters))
    ax.set_xticklabels([f'K={k}' for k in ks], fontsize=9,
                        rotation=45, ha='right')
    ax.set_yticks([])
    ax.set_ylabel('← 1D Embedding (similar clusters nearby) →', fontsize=10)
    ax.set_title(
        f'1D Cluster Embedding with Sankey Flow\n'
        f'K={ks[0]}..{ks[-1]}, seed={SEED}, '
        f'{N_TOTAL} data points, {embed_method} projection',
        fontsize=13, fontweight='bold')

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)

    for iter_idx, k in enumerate(ks):
        row = metrics_df[metrics_df['K'] == k]
        if len(row) > 0:
            sil = row.iloc[0]['Silhouette']
            ax.text(iter_idx, TOTAL_HEIGHT + 0.008,
                    f'Sil={sil:.3f}', ha='center', va='bottom',
                    fontsize=6, color='gray', style='italic')

    plt.tight_layout()
    plt.show()

# ═══════════════════════════════════════════════════════════════

#  Step 4: Interactive widgets — using Output widget for plot area

# ═══════════════════════════════════════════════════════════════

w_krange = widgets.IntRangeSlider(
    value=[K_MIN, min(K_MIN + 10, K_MAX)],
    min=K_MIN, max=K_MAX, step=1,
    description='K range:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='600px'))

w_method = widgets.ToggleButtons(
    options=EMBED_1D_METHODS,
    value='MDS',
    description='1D method:',
    style={'description_width': 'initial'})

w_sankey = widgets.Checkbox(
    value=True, description='Show Sankey flow',
    style={'description_width': 'initial'})

w_labels = widgets.Checkbox(
    value=True, description='Show labels',
    style={'description_width': 'initial'})

w_alpha = widgets.FloatSlider(
    value=0.20, min=0.05, max=0.60, step=0.05,
    description='Ribbon α:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px'))

# Dedicated output area — widgets live OUTSIDE this, so they survive redraws

out_plot = widgets.Output()

def redraw(*_):
    with out_plot:
        clear_output(wait=True)
        plot_1d_sankey(w_krange.value, w_method.value,
                       w_sankey.value, w_labels.value, w_alpha.value)

for w in [w_krange, w_method, w_sankey, w_labels, w_alpha]:
    w.observe(redraw, names='value')

display(widgets.VBox([
    widgets.HTML("<h3>📊 1D Cluster Embedding with Sankey Flow</h3>"),
    w_krange,
    widgets.HBox([w_method, w_sankey, w_labels, w_alpha]),
    out_plot    # ← plot goes here; widgets above stay untouched
]))

# Initial draw

redraw()

### Cell 8: Understand Cluster Profiles — What Makes Each Cluster Different?

After finding clusters, the natural question is: **"What characterizes each
cluster?"** This cell provides two complementary views of cluster centroid
profiles.

#### View 1: Line Chart (Z-scored)

Each cluster centroid is plotted as a **line across features** on the x-axis.
The y-axis shows **Z-scores** (standard deviations from the global mean):

| Z-score | Meaning |
|---------|---------|
| 0 | Exactly at the global average |
| +1 | One standard deviation above average |
| −1 | One standard deviation below average |

The **green zone** (above zero) and **red zone** (below zero) help identify
which features are unusually high or low for each cluster. Raw percentage values
are annotated near each point for reference.

#### View 2: Radar Chart (Min-Max normalized)

Each feature is **independently normalized** across clusters to [0, 1]:
- 0 = the cluster with the **lowest** value for that feature
- 1 = the cluster with the **highest** value

This removes the absolute scale and highlights **relative ranking** among
clusters. The polygon shapes reveal each cluster's "signature pattern."

> **When to use which?**
> - **Line chart**: Best for identifying features that are significantly
>   above/below the global mean (absolute interpretation)
> - **Radar chart**: Best for comparing cluster "shapes" — which cluster
>   is the "youngest"? "most female"? "most elderly"?

The table below the charts shows the Z-scored centroid values with conditional
formatting: green = above average, red = below average.

In [ ]:
# ── Cell 8 — Cluster Profiles: Line Chart (Z-scored) + Radar (Min-Max) ──

def show_cluster_profiles(k, chart_type='Both'):
    """
    Two views of cluster centroid profiles:
      - Line chart: Z-scored values (0 = global mean)
      - Radar chart: per-feature min-max normalized to [0, 1]

    """
    labels = cluster_assignments[k]
    cd = centroid_data[k]
    colors = cd['colors']

    # Centroids in Z-scored space
    centroids_z = cd['centroids']  # shape (k, n_features)

    # Centroids in raw space
    centroids_raw = np.array([
        df.loc[labels == c, FEATURE_COLS].mean().values
        for c in range(k)
    ])

    # ══════════════════════════════════════════════════════════
    #  Min-Max normalization per feature across clusters
    #  Each feature scaled independently to [0, 1]
    # ══════════════════════════════════════════════════════════
    centroids_minmax = centroids_raw.copy()
    feat_mins = centroids_raw.min(axis=0)
    feat_maxs = centroids_raw.max(axis=0)
    feat_ranges = feat_maxs - feat_mins

    for j in range(len(FEATURE_COLS)):
        if feat_ranges[j] > 1e-10:
            centroids_minmax[:, j] = (centroids_raw[:, j] - feat_mins[j]) / feat_ranges[j]
        else:
            centroids_minmax[:, j] = 0.5  # constant feature

    # ══════════════════════════════════════════════════════════
    show_line = chart_type in ('Both', 'Line chart (Z-score)')
    show_radar = chart_type in ('Both', 'Radar (Min-Max)')

    if show_line and show_radar:
        fig, (ax_line, ax_radar) = plt.subplots(
            1, 2, figsize=(20, 7),
            gridspec_kw={'width_ratios': [1.3, 1]},
            subplot_kw={'projection': None})
        # Radar needs polar projection — recreate just that axis
        pos = ax_radar.get_position()
        ax_radar.remove()
        ax_radar = fig.add_axes(pos, polar=True)
    elif show_line:
        fig, ax_line = plt.subplots(figsize=(14, 6))
        ax_radar = None
    elif show_radar:
        fig, ax_radar = plt.subplots(figsize=(8, 8), subplot_kw={'projection': 'polar'})
        ax_line = None
    else:
        return

    # ══════════════════════════════════════════════════════════
    #  LINE CHART (Z-scored)
    # ══════════════════════════════════════════════════════════
    if ax_line is not None:
        x_pos = np.arange(len(FEATURE_COLS))

        ax_line.axhline(y=0, color='black', linewidth=1.5, linestyle='-',
                        alpha=0.3, label='Global mean (Z=0)')

        for c in range(k):
            n_pts = (labels == c).sum()
            ax_line.plot(x_pos, centroids_z[c], 'o-',
                         color=colors[c], linewidth=2.5, markersize=8,
                         markeredgecolor='black', markeredgewidth=0.5,
                         label=f'Cl {c} (n={n_pts})', alpha=0.85)

            for j in range(len(FEATURE_COLS)):
                z_val = centroids_z[c, j]
                raw_val = centroids_raw[c, j]
                offset = 8 if (c % 2 == 0) else -12
                ax_line.annotate(f'{raw_val:.1f}',
                                 xy=(x_pos[j], z_val),
                                 xytext=(0, offset),
                                 textcoords='offset points',
                                 fontsize=5.5, color=colors[c], alpha=0.7,
                                 ha='center')

        # Shade zones
        ylim = ax_line.get_ylim()
        ax_line.axhspan(0, max(ylim[1], 0.1), alpha=0.03, color='green')
        ax_line.axhspan(min(ylim[0], -0.1), 0, alpha=0.03, color='red')
        ax_line.text(len(FEATURE_COLS) - 0.5, 0.1, 'Above avg',
                     fontsize=8, color='green', alpha=0.5, ha='right')
        ax_line.text(len(FEATURE_COLS) - 0.5, -0.2, 'Below avg',
                     fontsize=8, color='red', alpha=0.5, ha='right')

        ax_line.set_xticks(x_pos)
        ax_line.set_xticklabels(FEATURE_COLS, rotation=30, ha='right', fontsize=9)
        ax_line.set_ylabel('Z-score', fontsize=10)
        ax_line.set_title(f'Z-scored Centroid Profiles — K={k}',
                          fontsize=12, fontweight='bold')
        ax_line.grid(True, axis='y', alpha=0.3)
        ax_line.grid(True, axis='x', alpha=0.1)
        ax_line.legend(fontsize=6, loc='center left',
                       bbox_to_anchor=(1.01, 0.5) if ax_radar is None else (-0.02, -0.22),
                       framealpha=0.9,
                       ncol=max(1, k // 8))

    # ══════════════════════════════════════════════════════════
    #  RADAR CHART (per-feature Min-Max normalized)
    # ══════════════════════════════════════════════════════════
    if ax_radar is not None:
        n_feat = len(FEATURE_COLS)
        # Angles for each feature (evenly spaced around the circle)
        angles = np.linspace(0, 2 * np.pi, n_feat, endpoint=False).tolist()
        angles.append(angles[0])  # close the polygon

        # Short labels for the radar
        short_labels = [f.replace('pct_', '').replace('_', '\n') for f in FEATURE_COLS]

        ax_radar.set_theta_offset(np.pi / 2)
        ax_radar.set_theta_direction(-1)
        ax_radar.set_xticks(angles[:-1])
        ax_radar.set_xticklabels(short_labels, fontsize=8)

        # Radial ticks
        ax_radar.set_ylim(0, 1.05)
        ax_radar.set_yticks([0.0, 0.25, 0.5, 0.75, 1.0])
        ax_radar.set_yticklabels(['min', '25%', '50%', '75%', 'max'],
                                  fontsize=7, color='gray')
        ax_radar.yaxis.grid(True, alpha=0.3)

        for c in range(k):
            vals = centroids_minmax[c].tolist()
            vals.append(vals[0])  # close polygon
            n_pts = (labels == c).sum()

            ax_radar.plot(angles, vals, 'o-',
                          color=colors[c], linewidth=2, markersize=5,
                          markeredgecolor='black', markeredgewidth=0.3,
                          label=f'Cl {c} (n={n_pts})', alpha=0.8)
            ax_radar.fill(angles, vals, color=colors[c], alpha=0.08)

            # Annotate raw values at each vertex
            for j in range(n_feat):
                raw_val = centroids_raw[c, j]
                ax_radar.annotate(f'{raw_val:.1f}',
                                  xy=(angles[j], vals[j]),
                                  xytext=(3, 3),
                                  textcoords='offset points',
                                  fontsize=5, color=colors[c], alpha=0.7)

        ax_radar.set_title(f'Min-Max Normalized Profiles — K={k}\n'
                           f'(each feature scaled to its min..max across clusters)',
                           fontsize=10, fontweight='bold', y=1.08)
        ax_radar.legend(fontsize=6, loc='lower right',
                        bbox_to_anchor=(1.3, -0.05),
                        framealpha=0.9,
                        ncol=1 if k <= 12 else 2)

        # ── Add min/max reference annotations ──
        # Small table below radar showing actual min/max per feature
        info_lines = []
        for j, feat in enumerate(FEATURE_COLS):
            short = feat.replace('pct_', '')
            info_lines.append(f'{short}: [{feat_mins[j]:.1f} – {feat_maxs[j]:.1f}]')

        info_text = '  |  '.join(info_lines)
        fig.text(0.5 if ax_line is None else 0.75, 0.01,
                 f'Feature ranges: {info_text}',
                 fontsize=7, ha='center', color='gray', style='italic')

    plt.tight_layout()
    plt.subplots_adjust(bottom=0.06)
    plt.show()

    # ══════════════════════════════════════════════════════════
    #  Table: Z-scored values with cluster color indicators
    # ══════════════════════════════════════════════════════════
    profile_z = pd.DataFrame(centroids_z, columns=FEATURE_COLS)
    profile_z.index.name = 'Cluster'
    profile_raw = pd.DataFrame(centroids_raw, columns=[f'{c}_raw' for c in FEATURE_COLS])
    profile_mm = pd.DataFrame(centroids_minmax, columns=[f'{c}_mm' for c in FEATURE_COLS])
    profile_z['N'] = [(labels == c).sum() for c in range(k)]
    cols = ['N'] + FEATURE_COLS
    profile_z = profile_z[cols]

    def color_cells(val):
        if isinstance(val, (int, np.integer)):
            return ''
        if val > 0.5:
            return f'background-color: rgba(0,128,0,{min(abs(val)/3, 0.4):.2f})'
        elif val < -0.5:
            return f'background-color: rgba(200,0,0,{min(abs(val)/3, 0.4):.2f})'
        return ''

    def row_border(row):
        c_idx = int(row.name)
        col = colors[c_idx] if c_idx < len(colors) else '#999999'
        return [f'border-left: 5px solid {col}'] + [''] * (len(row) - 1)

    styled = (profile_z.style
              .map(color_cells)
              .apply(row_border, axis=1)
              .format(precision=2))
    display(HTML(f"<h4>Z-scored Centroid Values (K={k})</h4>"
                 f"<p style='font-size:11px; color:gray'>"
                 f"Green = above global mean, Red = below global mean</p>"))
    display(styled)


# ── Interactive widgets ──

widgets.interact(
    show_cluster_profiles,
    k=widgets.IntSlider(value=min(10, K_MAX), min=K_MIN, max=K_MAX,
                        description='K:',
                        style={'description_width': 'initial'},
                        layout=widgets.Layout(width='500px')),
    chart_type=widgets.ToggleButtons(
        options=['Both', 'Line chart (Z-score)', 'Radar (Min-Max)'],
        value='Both',
        description='View:',
        style={'description_width': 'initial'})
)

### Cell 9: Interactive Geographic Map — Where Are the Clusters?

This is the geographic exploration tool. For a selected K, it shows:

#### Left panel (2/3 width):

**Top: Choropleth Map**
- Each NUTS3 region is colored by its cluster assignment
- Colors come from the centroid projection (Cell 5b) — similar clusters have similar colors
- Country borders drawn in black for reference
- Quality metrics (Silhouette, Variance Explained, Davies-Bouldin) shown in the top-right corner

**Bottom: Centroid 2D Embedding**
- Same as the right panel of Cell 5b, but for the currently selected K only
- Circle size ∝ cluster population
- Gray lines connect nearest-neighbor centroids
- Helps understand the color logic: nearby centroids = similar colors

#### Reading the Centroid Embedding

The scatter plot below the map shows cluster centroids projected into 2D.

- **Circle size** ∝ number of regions in that cluster
- **Circle color** = same as on the map
- **Gray lines** connect each centroid to its **nearest neighbor in the

  original 6D feature space** (not in the 2D projection). This helps
  assess projection quality:
  - Short lines between nearby circles → projection is faithful
  - Long lines crossing the plot → some distances are distorted by
    the dimensionality reduction

#### Right panel (1/3 width):

**Z-scored Centroids Table** (shown first)
- Each row = one cluster; left border colored to match the map
- Values show how many standard deviations each cluster's mean is from the global mean
- Green cells = significantly above average; Red cells = significantly below average

**Raw Cluster Profiles Table** (shown second)
- Same structure but with actual percentage values
- Row backgrounds match cluster colors for easy cross-referencing with the map

#### Using the slider:

Move the K slider to instantly see how the geographic pattern changes.
Observe how:
- Low K → broad, continental-scale groupings (e.g., "Western vs. Eastern Europe")
- Medium K → national/regional patterns emerge
- High K → fine-grained local differences visible

> **A progress bar** appears during rendering because drawing ~1500 polygons
> takes a few seconds.

In [ ]:
# ── Cell 9 — Interactive Map with progress indicator ──────────

# ── Pre-merge shapefile with data ──

geo = nuts3_shp[[SHP_ID_COL, 'geometry']].copy()
geo = geo.rename(columns={SHP_ID_COL: 'ID_NUTS3'})
geo['ID_NUTS3'] = geo['ID_NUTS3'].astype(str)
df['ID_NUTS3'] = df['ID_NUTS3'].astype(str)
geo_merged = geo.merge(
    df[['ID_NUTS3', 'Name', 'ID_NUTS0', 'total_population'] + FEATURE_COLS],
    on='ID_NUTS3', how='left')
print(f"Geo-merged: {len(geo_merged)} shapes, "
      f"{geo_merged['Name'].notna().sum()} matched to data")

# ── Build cluster profile ──

def cluster_profile(k, labels):
    tmp = df[FEATURE_COLS].copy()
    tmp['Cluster'] = labels
    tmp['count'] = 1
    profile = tmp.groupby('Cluster').agg(
        N=('count', 'sum'),
        **{f: (f, 'mean') for f in FEATURE_COLS}
    ).round(2)
    return profile

# ── Widgets ──

slider_k = widgets.IntSlider(
    value=K_MIN, min=K_MIN, max=K_MAX, step=1,
    description='K (clusters):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px'))

out_plot = widgets.Output()
out_profile = widgets.Output()

# Progress bar + status label

progress_bar = widgets.IntProgress(
    value=0, min=0, max=100,
    bar_style='info',
    layout=widgets.Layout(width='500px', visibility='hidden'))
status_label = widgets.HTML(value='')

def set_progress(pct, msg=''):
    progress_bar.layout.visibility = 'visible'
    progress_bar.value = pct
    status_label.value = (
        f"<span style='font-size:12px; color:#555;'>"
        f"⏳ {msg}...</span>" if msg else '')

def clear_progress():
    progress_bar.layout.visibility = 'hidden'
    progress_bar.value = 0
    status_label.value = ''

def update_map(*_):
    import time
    t0 = time.time()
    k = slider_k.value
    labels = cluster_assignments[k]
    cd = centroid_data[k]
    colors = cd['colors']

    # ── Step 1: Prepare data ──
    set_progress(10, f'Preparing data for K={k}')
    label_df = pd.DataFrame({'ID_NUTS3': df['ID_NUTS3'], 'Cluster': labels})
    gdf = geo_merged.copy()
    gdf = gdf.merge(label_df, on='ID_NUTS3', how='left')
    gdf['Cluster'] = gdf['Cluster'].fillna(-1).astype(int)

    # ── Step 2: Draw map + centroid embedding ──
    set_progress(20, 'Drawing map')

    with out_plot:
        clear_output(wait=True)

        # height_ratios: map=2.5, centroid=2 (centroid is 2× bigger than before)
        fig = plt.figure(figsize=(14, 16))
        gs = fig.add_gridspec(2, 1, height_ratios=[2.5, 2], hspace=0.12)
        ax_map = fig.add_subplot(gs[0])
        ax_embed = fig.add_subplot(gs[1])

        set_progress(30, 'Rendering unmatched regions')
        unmatched = gdf[gdf['Cluster'] == -1]
        if len(unmatched) > 0:
            unmatched.plot(ax=ax_map, color='#E0E0E0',
                          edgecolor='#CCCCCC', linewidth=0.3)

        set_progress(40, 'Rendering clusters')
        matched = gdf[gdf['Cluster'] >= 0].copy()
        for c_idx in range(k):
            subset = matched[matched['Cluster'] == c_idx]
            if len(subset) > 0:
                subset.plot(ax=ax_map, color=colors[c_idx],
                            edgecolor='#999999', linewidth=0.3)
            set_progress(40 + int(30 * (c_idx + 1) / k),
                         f'Rendering cluster {c_idx+1}/{k}')

        # Legend
        from matplotlib.patches import Patch
        legend_patches = [
            Patch(facecolor=colors[c], edgecolor='black',
                  label=f'{c} (n={(labels==c).sum()})')
            for c in range(k)]
        ax_map.legend(handles=legend_patches, loc='lower left',
                      fontsize=6, framealpha=0.9, title='Cluster',
                      title_fontsize=8,
                      ncol=2 if k > 15 else 1)

        set_progress(72, 'Drawing country borders')
        nuts0_shp.boundary.plot(ax=ax_map, color='black', linewidth=1.0)

        # Metrics annotation
        row = metrics_df[metrics_df['K'] == k].iloc[0]
        info_text = (f"Silhouette: {row['Silhouette']:.4f}\n"
                     f"Var.Explained: {row['Variance_Explained']:.1f}%\n"
                     f"Davies-Bouldin: {row['Davies_Bouldin']:.4f}")
        ax_map.text(0.98, 0.98, info_text, transform=ax_map.transAxes,
                    fontsize=9, verticalalignment='top',
                    horizontalalignment='right',
                    bbox=dict(boxstyle='round', facecolor='lightyellow',
                              alpha=0.9))

        ax_map.set_title(
            f"NUTS3 Population Clusters — K={k}, seed={SEED}, data years {YEAR_MIN}–{YEAR_MAX}",
            fontsize=13, fontweight='bold')
        ax_map.set_axis_off()
        ax_map.set_xlim(-25, 45)
        ax_map.set_ylim(34, 72)

        # ── Centroid 2D embedding (larger) ──
        set_progress(78, 'Drawing centroid embedding')
        coords = cd['coords_2d']
        sizes = [(labels == c).sum() for c in range(k)]
        max_size = max(sizes) if max(sizes) > 0 else 1
        # Larger markers to fill the bigger subplot
        marker_sizes = [max(120, 700 * s / max_size) for s in sizes]

        for c in range(k):
            ax_embed.scatter(coords[c, 0], coords[c, 1],
                             c=colors[c], s=marker_sizes[c],
                             edgecolors='black', linewidths=1.0,
                             zorder=3)
            ax_embed.annotate(f'{c}', (coords[c, 0], coords[c, 1]),
                              ha='center', va='center',
                              fontsize=9, fontweight='bold')

        from scipy.spatial.distance import pdist, squareform as sqf
        dist_matrix = sqf(pdist(cd['centroids']))
        for i in range(k):
            nearest = np.argsort(dist_matrix[i])[1]
            ax_embed.plot(
                [coords[i, 0], coords[nearest, 0]],
                [coords[i, 1], coords[nearest, 1]],
                color='gray', alpha=0.3, linewidth=1.0, zorder=1)

        ax_embed.set_title(
            f'Centroid 2D embedding — K={k} (size ∝ cluster population)',
            fontsize=11, fontweight='bold')
        ax_embed.set_xlim(-0.1, 1.1)
        ax_embed.set_ylim(-0.1, 1.1)
        ax_embed.set_aspect('equal')
        ax_embed.grid(True, alpha=0.2)

        set_progress(88, 'Finalizing figure')
        plt.tight_layout()
        plt.show()

    # ══════════════════════════════════════════════════════
    #  RIGHT side: Z-scored centroids FIRST, then raw profiles
    # ══════════════════════════════════════════════════════
    set_progress(92, 'Building profile tables')
    with out_profile:
        clear_output(wait=True)

        centroids_z = cd['centroids']

        # ── Table 1: Z-scored centroids (shown first) ──
        profile_z = pd.DataFrame(centroids_z, columns=FEATURE_COLS)
        profile_z.index.name = 'Cluster'
        profile_z['N'] = [(labels == c).sum() for c in range(k)]
        cols_z = ['N'] + FEATURE_COLS
        profile_z = profile_z[cols_z]

        def z_color_cells(val):
            if isinstance(val, (int, np.integer)):
                return ''
            if val > 0.5:
                return (f'background-color: '
                        f'rgba(0,128,0,{min(abs(val)/3, 0.4):.2f})')
            elif val < -0.5:
                return (f'background-color: '
                        f'rgba(200,0,0,{min(abs(val)/3, 0.4):.2f})')
            return ''

        def row_border(row):
            c_idx = int(row.name)
            col = colors[c_idx] if c_idx < len(colors) else '#999999'
            return [f'border-left: 5px solid {col}'] + \
                   [''] * (len(row) - 1)

        styled_z = (profile_z.style
                    .map(z_color_cells)
                    .apply(row_border, axis=1)
                    .format(precision=2))
        display(HTML(
            f"<h4>Z-scored Centroids (K={k})</h4>"
            f"<p style='font-size:11px; color:gray'>"
            f"Green = above average, Red = below average</p>"))
        display(styled_z)

        # ── Table 2: Raw cluster profiles (shown second) ──
        profile = cluster_profile(k, labels)

        def color_rows(row):
            c_idx = int(row.name)
            bg = colors[c_idx] if c_idx < len(colors) else '#FFFFFF'
            r, g, b = (int(bg[1:3],16)/255,
                       int(bg[3:5],16)/255,
                       int(bg[5:7],16)/255)
            lum = 0.299*r + 0.587*g + 0.114*b
            fg = '#000000' if lum > 0.45 else '#FFFFFF'
            return [f'background-color: {bg}; color: {fg}'] * len(row)

        styled = (profile.style
                  .apply(color_rows, axis=1)
                  .format(precision=2))
        display(HTML(f"<h4>Cluster Profiles — raw values (K={k})</h4>"))
        display(styled)

    elapsed = time.time() - t0
    set_progress(100, f'Done in {elapsed:.1f}s')

    import threading
    def hide_later():
        import time as _t
        _t.sleep(2.0)
        clear_progress()
    threading.Thread(target=hide_later, daemon=True).start()


slider_k.observe(update_map, names='value')

# ═══════════════════════════════════════════════════════════

#  Layout

# ═══════════════════════════════════════════════════════════

display(widgets.VBox([
    widgets.HTML("<h3>🗺️ Select K to view cluster map</h3>"),
    widgets.HBox([slider_k, progress_bar, status_label]),
    widgets.HBox([
        out_plot,
        out_profile
    ], layout=widgets.Layout(
        display='flex',
        flex_flow='row',
        gap='15px',
        width='100%'
    ))
]))

# Initial render

update_map()

### Cell 10: Transitions FROM a Specific Cluster to a Later Iteration

**What this cell does:**

Given a source cluster at iteration K₁ (e.g., K=20, cluster 10), this cell tracks
where all its members end up at a later iteration K₂ (e.g., K=24). It answers:

> *"When we increase granularity, how does this cluster split — and what demographic
> differences drive the split?"*

Following Section 6.2 of the paper (cf. Figure 7b): the tool "highlights the
transition from 20.10 to all clusters at K = 24, creating a class attribute
whose values encode each destination."

**Outputs:**

| # | Output | Purpose |
|---|--------|---------|
| 1 | **Summary table** | Destination clusters ranked by member count, with percentages and bar indicators |
| 2 | **Sankey diagram** | Single source node → multiple destination nodes; band width = shared members |
| 3 | **Line chart (Z-scores)** | Source cluster profile (thick grey) overlaid with each destination sub-group profile (coloured); features on x-axis, Z-scores on y-axis |
| 4 | **Radar chart** | Same profiles as the line chart, displayed on a polar plot for shape comparison |
| 5 | **Choropleth map** | Regions of the source cluster colour-coded by their destination at K₂ |

**How to interpret the profile plots:**

The line chart and radar show the **Z-score profile of each destination sub-group**
(i.e., "members of K₁.C that end up in destination D"). The thick grey line is
the overall source cluster profile. Where coloured lines diverge from each other,
those are the **discriminating features** that explain *why* the cluster splits.

| Pattern in profiles | Interpretation |
|---------------------|----------------|
| All destination lines overlap closely | Split is geographic, not demographic — features don't explain it |
| Lines diverge on one feature (e.g., pct_65_plus) | That feature is the axis of differentiation at higher K |
| One sub-group matches the grey line, others deviate | The core remains; peripheral members peel off |

**How to interpret the flow patterns:**

| Pattern in Sankey/table | Interpretation |
|-------------------------|----------------|
| One dominant destination (>80%) | Cluster is stable — persists almost intact at higher K |
| Two roughly equal destinations | Clean split — the cluster contains two meaningful sub-groups |
| Many small destinations | Fragmentation — cluster coherence breaks down at higher K |
| One large + several tiny flows | Core persists; a few outlier regions get reassigned |

**Use case:** After identifying a cluster of interest on the map (e.g., "Ageing rural
Eastern Europe"), use this cell to check whether it remains a coherent unit at
higher K, and if it splits, *which features* differentiate the resulting sub-groups.

In [ ]:
# ── Cell 10 — Transitions FROM a specific cluster to a later iteration ──────
#
# Following Section 6.2 of the paper (cf. Figure 7b):
# Given a source cluster at K1, show where its members end up at K2,
# AND show the demographic profiles of each destination sub-group
# to interpret WHY the split occurs.

def show_transitions_from_cluster(k1, cluster_id, k2):
    """
    Track members of a specific cluster at K=k1, show their distribution
    across clusters at K=k2, and compare profiles of destination sub-groups.
    """
    labels_k1 = cluster_assignments[k1]
    labels_k2 = cluster_assignments[k2]

    # ── Identify source members ──
    source_mask = (labels_k1 == cluster_id)
    n_source = int(source_mask.sum())

    if n_source == 0:
        print(f"⚠ No members in K={k1}, cluster {cluster_id}")
        return

    # ── Count transitions to each destination ──
    dest_labels = labels_k2[source_mask]
    dest_clusters, counts = np.unique(dest_labels, return_counts=True)
    sort_idx = np.argsort(-counts)
    dest_clusters = dest_clusters[sort_idx]
    counts = counts[sort_idx]
    pcts = counts / n_source * 100

    # ── Print summary table ──
    print(f"\n{'═'*60}")
    print(f"  Transitions FROM  K={k1}.cluster{cluster_id}  ({n_source} members)")
    print(f"  TO iteration K={k2}  ({k2} clusters)")
    print(f"{'═'*60}")
    print(f"  {'Destination':<22} {'Count':>7} {'Percent':>9}")
    print(f"  {'─'*40}")
    for dc, cnt, pct in zip(dest_clusters, counts, pcts):
        bar = '█' * int(pct / 3)
        print(f"  K={k2}.cluster{dc:<8} {cnt:>7,} {pct:>7.1f}%  {bar}")

    # ── Colour palette ──
    palette = (
        ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd',
         '#8c564b','#e377c2','#7f7f7f','#bcbd22','#17becf'] * 5
    )

    # ── Build Sankey diagram ──
    n_dest = len(dest_clusters)
    source_indices = [0] * n_dest
    target_indices = list(range(1, n_dest + 1))

    node_labels = [f"K={k1}.C{cluster_id}<br>({n_source})"]
    node_colors = ['rgba(70,70,70,0.9)']

    for i, dc in enumerate(dest_clusters):
        node_labels.append(f"K={k2}.C{dc}<br>({counts[i]})")
        node_colors.append(palette[int(dc) % len(palette)])

    link_colors = [palette[int(dc) % len(palette)].replace('#', '')
                   for dc in dest_clusters]
    link_colors = [f"rgba({int(c[0:2],16)},{int(c[2:4],16)},{int(c[4:6],16)},0.45)"
                   for c in link_colors]

    fig = go.Figure(go.Sankey(
        node=dict(pad=20, thickness=25,
                  line=dict(color='black', width=0.5),
                  label=node_labels, color=node_colors),
        link=dict(source=source_indices, target=target_indices,
                  value=counts.tolist(), color=link_colors)
    ))
    fig.update_layout(
        title=dict(text=f"Transitions: K={k1}.cluster{cluster_id} → K={k2}",
                   font_size=14),
        height=max(350, 50 * n_dest), width=700,
        margin=dict(l=20, r=20, t=50, b=20)
    )
    fig.show()

    # ══════════════════════════════════════════════════════════════════════
    # ── PROFILE ANALYSIS: Z-score profiles of destination sub-groups ─────
    # ══════════════════════════════════════════════════════════════════════

    # Compute global mean/std for Z-score normalisation
    global_mean = df[FEATURE_COLS].mean()
    global_std = df[FEATURE_COLS].std()

    # Source cluster overall profile
    source_profile = (df.loc[source_mask, FEATURE_COLS].mean() - global_mean) / global_std

    # Destination sub-group profiles
    dest_profiles = {}
    for dc in dest_clusters:
        sub_mask = source_mask & (labels_k2 == dc)
        if sub_mask.sum() > 0:
            dest_profiles[dc] = (df.loc[sub_mask, FEATURE_COLS].mean() - global_mean) / global_std

    # ── Line chart (parallel coordinates style) ──
    fig_line, ax_line = plt.subplots(figsize=(10, 5))
    x_pos = range(len(FEATURE_COLS))

    # Plot source cluster as thick dark line
    ax_line.plot(x_pos, source_profile.values, 'k-', linewidth=3,
                 alpha=0.4, label=f'K{k1}.C{cluster_id} (all, n={n_source})',
                 zorder=1)

    # Plot each destination sub-group
    for i, dc in enumerate(dest_clusters):
        if dc in dest_profiles:
            n_sub = int((source_mask & (labels_k2 == dc)).sum())
            ax_line.plot(x_pos, dest_profiles[dc].values,
                         color=palette[int(dc) % len(palette)],
                         linewidth=2.5, marker='o', markersize=6,
                         label=f'→ K{k2}.C{dc} (n={n_sub})',
                         zorder=2)

    ax_line.axhline(0, color='grey', linewidth=0.8, linestyle='--', alpha=0.5)
    ax_line.set_xticks(x_pos)
    ax_line.set_xticklabels(FEATURE_COLS, rotation=35, ha='right', fontsize=9)
    ax_line.set_ylabel('Z-score', fontsize=11)
    ax_line.set_title(
        f'Profile comparison: K={k1}.C{cluster_id} split into K={k2} destinations',
        fontsize=12, fontweight='bold')
    ax_line.legend(loc='best', fontsize=8, framealpha=0.9)
    ax_line.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

    # ── Radar chart ──
    n_features = len(FEATURE_COLS)
    angles = np.linspace(0, 2 * np.pi, n_features, endpoint=False).tolist()
    angles += angles[:1]  # close the polygon

    fig_radar, ax_radar = plt.subplots(figsize=(7, 7),
                                        subplot_kw=dict(polar=True))

    # Source cluster
    values_src = source_profile.values.tolist() + [source_profile.values[0]]
    ax_radar.plot(angles, values_src, 'k-', linewidth=2, alpha=0.4,
                  label=f'K{k1}.C{cluster_id} (all)')
    ax_radar.fill(angles, values_src, color='grey', alpha=0.05)

    # Destination sub-groups
    for i, dc in enumerate(dest_clusters):
        if dc in dest_profiles:
            vals = dest_profiles[dc].values.tolist() + [dest_profiles[dc].values[0]]
            color = palette[int(dc) % len(palette)]
            ax_radar.plot(angles, vals, color=color, linewidth=2, marker='o',
                          markersize=4, label=f'→ K{k2}.C{dc}')
            ax_radar.fill(angles, vals, color=color, alpha=0.08)

    ax_radar.set_xticks(angles[:-1])
    ax_radar.set_xticklabels(FEATURE_COLS, fontsize=9)
    ax_radar.set_title(
        f'Radar: K={k1}.C{cluster_id} → K={k2} destinations',
        fontsize=11, fontweight='bold', pad=20)
    ax_radar.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=8)
    plt.tight_layout()
    plt.show()

    # ── Map: colour regions by destination cluster ──
    transition_col = f"trans_{k1}c{cluster_id}_to_{k2}"
    df[transition_col] = ''
    for dc in dest_clusters:
        mask = source_mask & (labels_k2 == dc)
        df.loc[mask, transition_col] = f"→ K{k2}.C{dc}"

    map_df = nuts3_shp.merge(
        df[['ID_NUTS3', transition_col]],
        left_on=SHP_ID_COL, right_on='ID_NUTS3', how='left'
    )
    map_df[transition_col] = map_df[transition_col].fillna('')

    fig_map, ax = plt.subplots(1, 1, figsize=(12, 8))
    map_df[map_df[transition_col] == ''].plot(
        ax=ax, color='#f0f0f0', edgecolor='#cccccc', linewidth=0.2)

    for i, dc in enumerate(dest_clusters):
        label = f"→ K{k2}.C{dc}"
        subset = map_df[map_df[transition_col] == label]
        if len(subset) > 0:
            subset.plot(ax=ax, color=palette[int(dc) % len(palette)],
                        edgecolor='#333333', linewidth=0.3, label=label)

    nuts0_shp.boundary.plot(ax=ax, color='black', linewidth=0.8)
    ax.set_xlim(-12, 45)
    ax.set_ylim(34, 72)
    ax.set_title(f"Where do members of K={k1}.cluster{cluster_id} go at K={k2}?",
                 fontsize=13, fontweight='bold')
    ax.legend(loc='lower left', fontsize=8, framealpha=0.9)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

    df.drop(columns=[transition_col], inplace=True)


# ── Interactive widget ──
w_k1_from = widgets.Dropdown(options=list(range(K_MIN, K_MAX+1)),
                              value=10, description='Source K:')
w_c_from = widgets.IntText(value=0, description='Cluster:',
                            layout=widgets.Layout(width='150px'))
w_k2_from = widgets.Dropdown(options=list(range(K_MIN, K_MAX+1)),
                              value=15, description='Target K:')
btn_from = widgets.Button(description='Show Transitions',
                           button_style='info', icon='arrow-right')
out_from = widgets.Output()

def on_btn_from(b):
    with out_from:
        clear_output(wait=True)
        k1 = w_k1_from.value
        c = w_c_from.value
        k2 = w_k2_from.value
        if c >= k1:
            print(f"⚠ Cluster index must be 0..{k1-1}")
            return
        show_transitions_from_cluster(k1, c, k2)

btn_from.on_click(on_btn_from)
display(widgets.HBox([w_k1_from, w_c_from, w_k2_from, btn_from]))
display(out_from)

### Cell 11: Transitions TO a Specific Cluster from an Earlier Iteration

**What this cell does:**

Given a target cluster at iteration K₂ (e.g., K=24, cluster 7), this cell traces
where its members came from at an earlier iteration K₁ (e.g., K=20). It answers:

> *"What is the composition of this cluster — and are its origins demographically
> homogeneous or does it merge distinct types?"*

This is the reverse perspective of Cell 10: instead of tracking *dispersion*
(one → many), we track *convergence* (many → one).

**Outputs:**

| # | Output | Purpose |
|---|--------|---------|
| 1 | **Summary table** | Origin clusters ranked by contribution, with percentages |
| 2 | **Sankey diagram** | Multiple origin nodes → single target node; band width = contribution |
| 3 | **Line chart (Z-scores)** | Target cluster profile (thick grey) overlaid with each origin sub-group profile (coloured) |
| 4 | **Radar chart** | Same profiles on a polar plot for visual shape comparison |
| 5 | **Choropleth map** | Regions of the target cluster colour-coded by their origin at K₁ |

**How to interpret the profile plots:**

Each coloured line represents the profile of **members that came from a specific
origin cluster**. If all origin sub-groups have similar profiles (lines overlap),
the target cluster is internally coherent regardless of origin. If profiles diverge,
the cluster merges demographically different populations.

| Pattern in profiles | Interpretation |
|---------------------|----------------|
| All origin lines overlap with the grey target line | Coherent cluster — all contributing origins are demographically similar |
| Origin lines diverge from each other | The cluster merges distinct types — may be an artefact of too-low K₂ |
| One dominant origin matches target perfectly | This cluster is essentially a renamed/relocated version of that origin |

**How to interpret the flow patterns:**

| Pattern in Sankey/table | Interpretation |
|-------------------------|----------------|
| Single dominant origin (>80%) | Direct descendant — a subset carved from one parent |
| Two origins with ~50/50 split | Merger — this cluster unifies regions from two previously separate groups |
| Many small origins | Novel grouping — K₂ found structure that cuts across K₁ boundaries |

**Use case:** When a new cluster appears at higher K, use this cell to understand
whether it represents a genuine new archetype (drawn from multiple sources) or
simply a sub-division of an existing group. The profiles reveal whether the
contributing origins are demographically consistent (supporting the cluster's validity)
or contradictory (suggesting the grouping may be unstable). This directly supports
Task T3 from the paper: assessing whether the additional cluster adds interpretive value.

In [ ]:
# ── Cell 11 — Transitions TO a specific cluster from an earlier iteration ────
#
# The reverse of Cell 10: given a target cluster at K2, show where its
# members came from at K1, AND compare the profiles of each origin sub-group
# to understand what demographic types converged into this cluster.

def show_transitions_to_cluster(k1, k2, cluster_id):
    """
    For members of cluster `cluster_id` at K=k2, show which clusters
    they belonged to at K=k1, and compare their profiles.
    """
    labels_k1 = cluster_assignments[k1]
    labels_k2 = cluster_assignments[k2]

    # ── Identify target members ──
    target_mask = (labels_k2 == cluster_id)
    n_target = int(target_mask.sum())

    if n_target == 0:
        print(f"⚠ No members in K={k2}, cluster {cluster_id}")
        return

    # ── Count origins from k1 ──
    origin_labels = labels_k1[target_mask]
    origin_clusters, counts = np.unique(origin_labels, return_counts=True)
    sort_idx = np.argsort(-counts)
    origin_clusters = origin_clusters[sort_idx]
    counts = counts[sort_idx]
    pcts = counts / n_target * 100

    # ── Print summary table ──
    print(f"\n{'═'*60}")
    print(f"  Transitions TO  K={k2}.cluster{cluster_id}  ({n_target} members)")
    print(f"  FROM iteration K={k1}  ({k1} clusters)")
    print(f"{'═'*60}")
    print(f"  {'Origin':<22} {'Count':>7} {'Percent':>9}")
    print(f"  {'─'*40}")
    for oc, cnt, pct in zip(origin_clusters, counts, pcts):
        bar = '█' * int(pct / 3)
        print(f"  K={k1}.cluster{oc:<8} {cnt:>7,} {pct:>7.1f}%  {bar}")

    # ── Colour palette ──
    palette = (
        ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd',
         '#8c564b','#e377c2','#7f7f7f','#bcbd22','#17becf'] * 5
    )

    # ── Build Sankey diagram ──
    n_orig = len(origin_clusters)
    source_indices = list(range(n_orig))
    target_indices = [n_orig] * n_orig

    node_labels = []
    node_colors = []
    for i, oc in enumerate(origin_clusters):
        node_labels.append(f"K={k1}.C{oc}<br>({counts[i]})")
        node_colors.append(palette[int(oc) % len(palette)])

    node_labels.append(f"K={k2}.C{cluster_id}<br>({n_target})")
    node_colors.append('rgba(70,70,70,0.9)')

    link_colors = [palette[int(oc) % len(palette)].replace('#', '')
                   for oc in origin_clusters]
    link_colors = [f"rgba({int(c[0:2],16)},{int(c[2:4],16)},{int(c[4:6],16)},0.45)"
                   for c in link_colors]

    fig = go.Figure(go.Sankey(
        node=dict(pad=20, thickness=25,
                  line=dict(color='black', width=0.5),
                  label=node_labels, color=node_colors),
        link=dict(source=source_indices, target=target_indices,
                  value=counts.tolist(), color=link_colors)
    ))
    fig.update_layout(
        title=dict(text=f"Origins: K={k1} → K={k2}.cluster{cluster_id}",
                   font_size=14),
        height=max(350, 50 * n_orig), width=700,
        margin=dict(l=20, r=20, t=50, b=20)
    )
    fig.show()

    # ══════════════════════════════════════════════════════════════════════
    # ── PROFILE ANALYSIS: Z-score profiles of origin sub-groups ──────────
    # ══════════════════════════════════════════════════════════════════════

    global_mean = df[FEATURE_COLS].mean()
    global_std = df[FEATURE_COLS].std()

    # Target cluster overall profile
    target_profile = (df.loc[target_mask, FEATURE_COLS].mean() - global_mean) / global_std

    # Origin sub-group profiles (members of K2.C that came from each K1 cluster)
    origin_profiles = {}
    for oc in origin_clusters:
        sub_mask = target_mask & (labels_k1 == oc)
        if sub_mask.sum() > 0:
            origin_profiles[oc] = (df.loc[sub_mask, FEATURE_COLS].mean() - global_mean) / global_std

    # ── Line chart (parallel coordinates style) ──
    fig_line, ax_line = plt.subplots(figsize=(10, 5))
    x_pos = range(len(FEATURE_COLS))

    # Target cluster as thick dark line
    ax_line.plot(x_pos, target_profile.values, 'k-', linewidth=3,
                 alpha=0.4, label=f'K{k2}.C{cluster_id} (all, n={n_target})',
                 zorder=1)

    # Each origin sub-group
    for i, oc in enumerate(origin_clusters):
        if oc in origin_profiles:
            n_sub = int((target_mask & (labels_k1 == oc)).sum())
            ax_line.plot(x_pos, origin_profiles[oc].values,
                         color=palette[int(oc) % len(palette)],
                         linewidth=2.5, marker='o', markersize=6,
                         label=f'← K{k1}.C{oc} (n={n_sub})',
                         zorder=2)

    ax_line.axhline(0, color='grey', linewidth=0.8, linestyle='--', alpha=0.5)
    ax_line.set_xticks(x_pos)
    ax_line.set_xticklabels(FEATURE_COLS, rotation=35, ha='right', fontsize=9)
    ax_line.set_ylabel('Z-score', fontsize=11)
    ax_line.set_title(
        f'Profile comparison: origins merging into K={k2}.C{cluster_id}',
        fontsize=12, fontweight='bold')
    ax_line.legend(loc='best', fontsize=8, framealpha=0.9)
    ax_line.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

    # ── Radar chart ──
    n_features = len(FEATURE_COLS)
    angles = np.linspace(0, 2 * np.pi, n_features, endpoint=False).tolist()
    angles += angles[:1]

    fig_radar, ax_radar = plt.subplots(figsize=(7, 7),
                                        subplot_kw=dict(polar=True))

    # Target cluster
    values_tgt = target_profile.values.tolist() + [target_profile.values[0]]
    ax_radar.plot(angles, values_tgt, 'k-', linewidth=2, alpha=0.4,
                  label=f'K{k2}.C{cluster_id} (all)')
    ax_radar.fill(angles, values_tgt, color='grey', alpha=0.05)

    # Origin sub-groups
    for i, oc in enumerate(origin_clusters):
        if oc in origin_profiles:
            vals = origin_profiles[oc].values.tolist() + [origin_profiles[oc].values[0]]
            color = palette[int(oc) % len(palette)]
            ax_radar.plot(angles, vals, color=color, linewidth=2, marker='o',
                          markersize=4, label=f'← K{k1}.C{oc}')
            ax_radar.fill(angles, vals, color=color, alpha=0.08)

    ax_radar.set_xticks(angles[:-1])
    ax_radar.set_xticklabels(FEATURE_COLS, fontsize=9)
    ax_radar.set_title(
        f'Radar: origins → K={k2}.C{cluster_id}',
        fontsize=11, fontweight='bold', pad=20)
    ax_radar.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=8)
    plt.tight_layout()
    plt.show()

    # ── Map: colour regions by origin cluster ──
    transition_col = f"trans_{k1}_to_{k2}c{cluster_id}"
    df[transition_col] = ''
    for oc in origin_clusters:
        mask = target_mask & (labels_k1 == oc)
        df.loc[mask, transition_col] = f"← K{k1}.C{oc}"

    map_df = nuts3_shp.merge(
        df[['ID_NUTS3', transition_col]],
        left_on=SHP_ID_COL, right_on='ID_NUTS3', how='left'
    )
    map_df[transition_col] = map_df[transition_col].fillna('')

    fig_map, ax = plt.subplots(1, 1, figsize=(12, 8))
    map_df[map_df[transition_col] == ''].plot(
        ax=ax, color='#f0f0f0', edgecolor='#cccccc', linewidth=0.2)

    for i, oc in enumerate(origin_clusters):
        label = f"← K{k1}.C{oc}"
        subset = map_df[map_df[transition_col] == label]
        if len(subset) > 0:
            subset.plot(ax=ax, color=palette[int(oc) % len(palette)],
                        edgecolor='#333333', linewidth=0.3, label=label)

    nuts0_shp.boundary.plot(ax=ax, color='black', linewidth=0.8)
    ax.set_xlim(-12, 45)
    ax.set_ylim(34, 72)
    ax.set_title(f"Where do members of K={k2}.cluster{cluster_id} come from at K={k1}?",
                 fontsize=13, fontweight='bold')
    ax.legend(loc='lower left', fontsize=8, framealpha=0.9)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

    df.drop(columns=[transition_col], inplace=True)


# ── Interactive widget ──
w_k1_to = widgets.Dropdown(options=list(range(K_MIN, K_MAX+1)),
                            value=10, description='Source K:')
w_k2_to = widgets.Dropdown(options=list(range(K_MIN, K_MAX+1)),
                            value=15, description='Target K:')
w_c_to = widgets.IntText(value=0, description='Cluster:',
                          layout=widgets.Layout(width='150px'))
btn_to = widgets.Button(description='Show Origins',
                         button_style='warning', icon='arrow-left')
out_to = widgets.Output()

def on_btn_to(b):
    with out_to:
        clear_output(wait=True)
        k1 = w_k1_to.value
        k2 = w_k2_to.value
        c = w_c_to.value
        if c >= k2:
            print(f"⚠ Cluster index must be 0..{k2-1}")
            return
        show_transitions_to_cluster(k1, k2, c)

btn_to.on_click(on_btn_to)
display(widgets.HBox([w_k1_to, w_k2_to, w_c_to, btn_to]))
display(out_to)

### Cell 12: Transitions Between Two Specific Clusters (Pair Analysis)

**What this cell does:**

Given a specific source cluster (K₁.C₁) and a specific target cluster (K₂.C₂),
this cell performs the **three-class decomposition** described in Section 6.2 of the
paper (cf. Figure 7a) and then profiles each class to explain *what makes members
stay, leave, or join*.

| Class | Definition | Colour |
|-------|-----------|--------|
| **Shared (persistent core)** | Regions in *both* K₁.C₁ and K₂.C₂ | 🟢 Green |
| **Source only (left behind)** | Regions in K₁.C₁ but *not* in K₂.C₂ | 🔴 Red |
| **Target only (newly joined)** | Regions in K₂.C₂ but *not* in K₁.C₁ | 🔵 Blue |

It answers:

> *"How much do these two clusters overlap — and what demographic features
> distinguish the persistent core from those that depart or arrive?"*

**Outputs:**

| # | Output | Purpose |
|---|--------|---------|
| 1 | **Summary statistics** | Sizes, Jaccard similarity, retention rate, purity |
| 2 | **Bar chart** | Three-class counts side by side |
| 3 | **Line chart (Z-scores)** | Profiles of all three classes (solid coloured) + full source/target cluster profiles (dashed grey) for reference |
| 4 | **Radar chart** | Same three-class profiles on a polar plot |
| 5 | **ΔZ-score bar chart: "left behind" vs. core** | Per-feature difference — red bars show features where departed members score higher than the core |
| 6 | **ΔZ-score bar chart: "newly joined" vs. core** | Per-feature difference — blue bars show features where new members score higher than the core |
| 7 | **Choropleth map** | Three-colour map: green = core, red = departed, blue = joined |

**How to interpret the profile plots (line chart & radar):**

The three solid lines show what each sub-group "looks like" demographically.
The dashed lines show the full source and target cluster profiles for context.

| Pattern | Interpretation |
|---------|----------------|
| Green (core) matches both dashed lines closely | The transition preserves the cluster's identity — stable core |
| Red (left) deviates strongly on specific features | Those features explain *why* those regions no longer fit at K₂ |
| Blue (joined) deviates from green on specific features | Those features explain *what new demographic type* the cluster absorbs |
| Green sits between red and blue | The core is the "average" — the extremes get reassigned |

**How to interpret the ΔZ-score bar charts:**

These directly answer: *"What is different about the regions that leave/join
compared to the persistent core?"*

| Bar direction | Meaning |
|---------------|---------|
| Tall red bar (positive) in "left vs. core" | Departed regions score *higher* on that feature than the core |
| Tall red bar (negative) in "left vs. core" | Departed regions score *lower* on that feature than the core |
| Near-zero bars everywhere | No clear demographic driver — the split may be geographic rather than feature-based |

**Key metrics:**

| Metric | Formula | Meaning |
|--------|---------|---------|
| Retention rate | shared / source size | What fraction of K₁.C₁ persists into K₂.C₂ |
| Purity | shared / target size | What fraction of K₂.C₂ already existed in K₁.C₁ |
| Jaccard index | shared / (source + target − shared) | Overall set similarity (0 = disjoint, 1 = identical) |

**Use case:** After Cell 10 shows that K=20.C10 splits into K=24.C7 and K=24.C12,
use this cell to inspect *each* leg of the split individually. The profile charts
reveal whether the split is driven by a clear demographic axis (e.g., age structure)
or is merely geographic. The ΔZ-score bar charts isolate the discriminating
features at a glance, replacing the parallel-coordinates approach used in the paper
with a notebook-native equivalent optimised for scrolling analysis workflows.

In [ ]:
# ── Cell 12 — Specific transition between two clusters (three-class) ─────────
#
# Following Section 6.2 of the paper (cf. Figure 7a):
# Three-class decomposition + profile comparison of Shared / Source-only /
# Target-only sub-groups to interpret what distinguishes members that persist
# from those that leave or join.

def show_transition_between(k1, c1, k2, c2):
    """
    Three-class decomposition with profile analysis.
    """
    labels_k1 = cluster_assignments[k1]
    labels_k2 = cluster_assignments[k2]

    mask_c1 = (labels_k1 == c1)
    mask_c2 = (labels_k2 == c2)

    shared = mask_c1 & mask_c2
    source_only = mask_c1 & ~mask_c2
    target_only = ~mask_c1 & mask_c2

    n_c1 = int(mask_c1.sum())
    n_c2 = int(mask_c2.sum())
    n_shared = int(shared.sum())
    n_source_only = int(source_only.sum())
    n_target_only = int(target_only.sum())

    if n_c1 == 0:
        print(f"⚠ No members in K={k1}, cluster {c1}")
        return
    if n_c2 == 0:
        print(f"⚠ No members in K={k2}, cluster {c2}")
        return

    # ── Summary ──
    jaccard = n_shared / (n_c1 + n_c2 - n_shared) if (n_c1 + n_c2 - n_shared) > 0 else 0
    retention = n_shared / n_c1 * 100 if n_c1 > 0 else 0
    purity = n_shared / n_c2 * 100 if n_c2 > 0 else 0

    print(f"\n{'═'*65}")
    print(f"  Transition: K={k1}.cluster{c1} → K={k2}.cluster{c2}")
    print(f"{'═'*65}")
    print(f"  K={k1}.cluster{c1} size:  {n_c1:,}")
    print(f"  K={k2}.cluster{c2} size:  {n_c2:,}")
    print(f"  ─────────────────────────────────────")
    print(f"  Shared (persistent core):   {n_shared:>6,}  "
          f"({retention:.1f}% of source, {purity:.1f}% of target)")
    print(f"  Source only (left behind):  {n_source_only:>6,}  "
          f"({n_source_only/n_c1*100:.1f}% of source)")
    print(f"  Target only (newly joined): {n_target_only:>6,}  "
          f"({n_target_only/n_c2*100:.1f}% of target)")
    print(f"  ─────────────────────────────────────")
    print(f"  Jaccard similarity:         {jaccard:.3f}")
    print(f"  Retention rate:             {retention:.1f}%")
    print(f"  Purity (target):            {purity:.1f}%")

    # ── Bar chart ──
    fig = go.Figure()
    categories = ['Source only<br>(left behind)', 'Shared<br>(core)',
                  'Target only<br>(joined)']
    values = [n_source_only, n_shared, n_target_only]
    colors_bar = ['#d62728', '#2ca02c', '#1f77b4']

    fig.add_trace(go.Bar(
        x=categories, y=values,
        marker_color=colors_bar,
        text=[f"{v:,}<br>({v/max(n_c1,n_c2)*100:.0f}%)" for v in values],
        textposition='outside'
    ))
    fig.update_layout(
        title=f"K={k1}.C{c1} → K={k2}.C{c2}  |  Jaccard = {jaccard:.3f}",
        yaxis_title="Number of regions",
        height=350, width=550,
        margin=dict(t=60, b=40)
    )
    fig.show()

    # ══════════════════════════════════════════════════════════════════════
    # ── PROFILE ANALYSIS: three-class Z-score profiles ───────────────────
    # ══════════════════════════════════════════════════════════════════════

    global_mean = df[FEATURE_COLS].mean()
    global_std = df[FEATURE_COLS].std()

    profiles = {}
    class_info = {
        'Shared (core)': {'mask': shared, 'n': n_shared, 'color': '#2ca02c'},
        f'K{k1}.C{c1} only (left)': {'mask': source_only, 'n': n_source_only, 'color': '#d62728'},
        f'K{k2}.C{c2} only (joined)': {'mask': target_only, 'n': n_target_only, 'color': '#1f77b4'},
    }

    # Also add full source and target profiles for reference
    ref_info = {
        f'K{k1}.C{c1} (full)': {'mask': mask_c1, 'n': n_c1, 'color': '#555555'},
        f'K{k2}.C{c2} (full)': {'mask': mask_c2, 'n': n_c2, 'color': '#999999'},
    }

    for label, info in {**class_info, **ref_info}.items():
        if info['n'] > 0:
            profiles[label] = {
                'z': (df.loc[info['mask'], FEATURE_COLS].mean() - global_mean) / global_std,
                'n': info['n'],
                'color': info['color']
            }

    # ── Line chart ──
    fig_line, ax_line = plt.subplots(figsize=(11, 5))
    x_pos = range(len(FEATURE_COLS))

    # Reference lines (full clusters) — dashed
    for label in ref_info:
        if label in profiles:
            ax_line.plot(x_pos, profiles[label]['z'].values,
                         color=profiles[label]['color'],
                         linewidth=2, linestyle='--', alpha=0.5,
                         label=f"{label} (n={profiles[label]['n']})",
                         zorder=1)

    # Three-class lines — solid
    for label in class_info:
        if label in profiles and profiles[label]['n'] > 0:
            ax_line.plot(x_pos, profiles[label]['z'].values,
                         color=profiles[label]['color'],
                         linewidth=2.8, marker='o', markersize=7,
                         label=f"{label} (n={profiles[label]['n']})",
                         zorder=2)

    ax_line.axhline(0, color='grey', linewidth=0.8, linestyle='--', alpha=0.5)
    ax_line.set_xticks(x_pos)
    ax_line.set_xticklabels(FEATURE_COLS, rotation=35, ha='right', fontsize=9)
    ax_line.set_ylabel('Z-score', fontsize=11)
    ax_line.set_title(
        f'Profile comparison: K={k1}.C{c1} → K={k2}.C{c2} (three-class decomposition)',
        fontsize=12, fontweight='bold')
    ax_line.legend(loc='best', fontsize=8, framealpha=0.9)
    ax_line.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

    # ── Radar chart ──
    n_features = len(FEATURE_COLS)
    angles = np.linspace(0, 2 * np.pi, n_features, endpoint=False).tolist()
    angles += angles[:1]

    fig_radar, ax_radar = plt.subplots(figsize=(7, 7),
                                        subplot_kw=dict(polar=True))

    # Three-class profiles
    for label in class_info:
        if label in profiles and profiles[label]['n'] > 0:
            vals = profiles[label]['z'].values.tolist() + [profiles[label]['z'].values[0]]
            ax_radar.plot(angles, vals, color=profiles[label]['color'],
                          linewidth=2.5, marker='o', markersize=4, label=label)
            ax_radar.fill(angles, vals, color=profiles[label]['color'], alpha=0.08)

    # Reference: full clusters (dashed)
    for label in ref_info:
        if label in profiles:
            vals = profiles[label]['z'].values.tolist() + [profiles[label]['z'].values[0]]
            ax_radar.plot(angles, vals, color=profiles[label]['color'],
                          linewidth=1.5, linestyle='--', alpha=0.5, label=label)

    ax_radar.set_xticks(angles[:-1])
    ax_radar.set_xticklabels(FEATURE_COLS, fontsize=9)
    ax_radar.set_title(
        f'Radar: K={k1}.C{c1} → K={k2}.C{c2}',
        fontsize=11, fontweight='bold', pad=20)
    ax_radar.legend(loc='upper right', bbox_to_anchor=(1.4, 1.15), fontsize=8)
    plt.tight_layout()
    plt.show()

    # ── Difference chart: what distinguishes "left behind" from "shared" ──
    if n_source_only > 0 and n_shared > 0:
        diff_left = profiles[f'K{k1}.C{c1} only (left)']['z'] - profiles['Shared (core)']['z']

        fig_diff, ax_diff = plt.subplots(figsize=(10, 4))
        colors_diff = ['#d62728' if v > 0 else '#2ca02c' for v in diff_left.values]
        bars = ax_diff.bar(x_pos, diff_left.values, color=colors_diff, alpha=0.8,
                           edgecolor='black', linewidth=0.5)
        ax_diff.axhline(0, color='black', linewidth=0.8)
        ax_diff.set_xticks(x_pos)
        ax_diff.set_xticklabels(FEATURE_COLS, rotation=35, ha='right', fontsize=9)
        ax_diff.set_ylabel('ΔZ-score', fontsize=11)
        ax_diff.set_title(
            f'What distinguishes "left behind" from "persistent core"?\n'
            f'(Red bars = features where departed members score higher)',
            fontsize=11, fontweight='bold')
        ax_diff.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        plt.show()

    # ── Difference chart: what distinguishes "newly joined" from "shared" ──
    if n_target_only > 0 and n_shared > 0:
        diff_joined = profiles[f'K{k2}.C{c2} only (joined)']['z'] - profiles['Shared (core)']['z']

        fig_diff2, ax_diff2 = plt.subplots(figsize=(10, 4))
        colors_diff2 = ['#1f77b4' if v > 0 else '#2ca02c' for v in diff_joined.values]
        bars2 = ax_diff2.bar(x_pos, diff_joined.values, color=colors_diff2, alpha=0.8,
                             edgecolor='black', linewidth=0.5)
        ax_diff2.axhline(0, color='black', linewidth=0.8)
        ax_diff2.set_xticks(x_pos)
        ax_diff2.set_xticklabels(FEATURE_COLS, rotation=35, ha='right', fontsize=9)
        ax_diff2.set_ylabel('ΔZ-score', fontsize=11)
        ax_diff2.set_title(
            f'What distinguishes "newly joined" from "persistent core"?\n'
            f'(Blue bars = features where new members score higher)',
            fontsize=11, fontweight='bold')
        ax_diff2.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        plt.show()

    # ── Map with three-class colouring ──
    transition_col = f"trans_{k1}c{c1}_{k2}c{c2}"
    df[transition_col] = 'Other'
    df.loc[shared, transition_col] = 'Shared (core)'
    df.loc[source_only, transition_col] = f'K{k1}.C{c1} only'
    df.loc[target_only, transition_col] = f'K{k2}.C{c2} only'

    map_df = nuts3_shp.merge(
        df[['ID_NUTS3', transition_col]],
        left_on=SHP_ID_COL, right_on='ID_NUTS3', how='left'
    )
    map_df[transition_col] = map_df[transition_col].fillna('Other')

    fig_map, ax = plt.subplots(1, 1, figsize=(13, 8))

    map_df[map_df[transition_col] == 'Other'].plot(
        ax=ax, color='#f5f5f5', edgecolor='#dddddd', linewidth=0.15)

    class_colors = {
        f'K{k1}.C{c1} only': '#d62728',
        'Shared (core)': '#2ca02c',
        f'K{k2}.C{c2} only': '#1f77b4'
    }

    for cls, color in class_colors.items():
        subset = map_df[map_df[transition_col] == cls]
        if len(subset) > 0:
            subset.plot(ax=ax, color=color, edgecolor='#333333',
                        linewidth=0.3, label=cls)

    nuts0_shp.boundary.plot(ax=ax, color='black', linewidth=0.8)
    ax.set_xlim(-12, 45)
    ax.set_ylim(34, 72)
    ax.set_title(
        f"Transition: K={k1}.C{c1} → K={k2}.C{c2}\n"
        f"Green = persistent core | Red = left behind | Blue = newly joined",
        fontsize=12, fontweight='bold')
    ax.legend(loc='lower left', fontsize=9, framealpha=0.9)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

    df.drop(columns=[transition_col], inplace=True)


# ── Interactive widget ──
w_k1_pair = widgets.Dropdown(options=list(range(K_MIN, K_MAX+1)),
                              value=10, description='K₁:')
w_c1_pair = widgets.IntText(value=0, description='C₁:',
                             layout=widgets.Layout(width='120px'))
w_k2_pair = widgets.Dropdown(options=list(range(K_MIN, K_MAX+1)),
                              value=15, description='K₂:')
w_c2_pair = widgets.IntText(value=0, description='C₂:',
                             layout=widgets.Layout(width='120px'))
btn_pair = widgets.Button(description='Show Transition',
                           button_style='success', icon='exchange')
out_pair = widgets.Output()

def on_btn_pair(b):
    with out_pair:
        clear_output(wait=True)
        k1, c1 = w_k1_pair.value, w_c1_pair.value
        k2, c2 = w_k2_pair.value, w_c2_pair.value
        if c1 >= k1:
            print(f"⚠ C₁ must be 0..{k1-1}")
            return
        if c2 >= k2:
            print(f"⚠ C₂ must be 0..{k2-1}")
            return
        show_transition_between(k1, c1, k2, c2)

btn_pair.on_click(on_btn_pair)
display(widgets.HBox([w_k1_pair, w_c1_pair,
                      widgets.Label('  →  '),
                      w_k2_pair, w_c2_pair, btn_pair]))
display(out_pair)

### Cell 13: Static HTML Export (All Visualizations Including Transitions)

This cell generates **static versions** of all key visualizations for HTML export.
Since Cells 10–12 use interactive widgets (buttons, dropdowns) that don't survive
`nbconvert`, this cell **automatically selects and renders representative transition
examples** alongside the metrics, Sankey, maps, and profiles.

**What is included:**

| Section | Source Cell(s) | Content |
|---------|---------------|---------|
| Quality Metrics | Cell 5 | Line plots of SSE, Silhouette, etc. vs. K |
| 1D Sankey Flow | Cell 7 | Full-range cluster evolution diagram |
| Maps + Profiles | Cells 8–9 | Choropleth maps, centroid embeddings, Z-score tables |
| **Transition FROM** | Cell 10 | Tracks the largest cluster at K₁ → where it goes at K₂ |
| **Transition TO** | Cell 11 | Traces origins of the largest cluster at K₂ ← from K₁ |
| **Pair Transition** | Cell 12 | Three-class decomposition for an auto-detected cluster pair |

**How transitions are auto-selected:**

Since we can't use interactive widgets in a static export, the cell applies
heuristics to pick *informative* transitions:

1. **FROM (Cell 10 logic):** Picks the largest cluster at a lower K (e.g., K=10)
   and shows where its members go at a higher K (e.g., K=15). Large clusters
   that split reveal the most structure.

2. **TO (Cell 11 logic):** Picks the largest cluster at a higher K (e.g., K=15)
   and traces where its members came from at a lower K (e.g., K=10). This shows
   whether the cluster is a direct descendant or a merger.

3. **Pair (Cell 12 logic):** Finds the cluster pair across two K values with the
   highest Jaccard similarity (strongest continuity) and shows its three-class
   decomposition — revealing which members persist vs. leave vs. join.

**Why a static export?**
- Jupyter widgets (`ipywidgets`) don't render in exported HTML files
- This cell calls the same analysis functions from Cells 10–12 with pre-selected
  parameters, producing matplotlib/plotly figures that embed directly in HTML
- All Plotly charts retain zoom/hover/pan in the exported file (client-side JS)

> **Output:** Run this cell, then export via  
> `File → Download as → HTML (.html)` or  
> `jupyter nbconvert --to html --execute notebook.ipynb`

In [ ]:
# ── Cell 13 — Static Export: All Key Visualizations + Transitions ────────────
#
# Generates STATIC versions of all interactive plots (including transitions)
# for a set of selected K values. These survive HTML export.
# Run this cell BEFORE exporting to HTML.

from matplotlib.patches import Patch
from scipy.spatial.distance import pdist, squareform as sqf
from matplotlib.gridspec import GridSpec

# ═══════════════════════════════════════════════════════════════
#  Configuration: which K values to include in the static export
# ═══════════════════════════════════════════════════════════════

# Auto-select: best silhouette, best DB, and a few evenly spaced
best_sil_k = int(metrics_df.loc[metrics_df['Silhouette'].idxmax(), 'K'])
best_db_k = int(metrics_df.loc[metrics_df['Davies_Bouldin'].idxmin(), 'K'])
EXPORT_KS = sorted(set([K_MIN, best_sil_k, best_db_k,
                         (K_MIN + K_MAX) // 2, K_MAX]))

print(f"Static export for K = {EXPORT_KS}")
print(f"  (best Silhouette: K={best_sil_k}, "
      f"best Davies-Bouldin: K={best_db_k})")
print("=" * 70)

# ═══════════════════════════════════════════════════════════════
#  Part 1: Metrics Charts
# ═══════════════════════════════════════════════════════════════

fig_met, axes_met = plt.subplots(3, 2, figsize=(14, 10))
axes_flat = axes_met.flatten()

for i, (metric, info) in enumerate(METRIC_INFO.items()):
    ax = axes_flat[i]
    ax.plot(metrics_df['K'], metrics_df[metric],
            'o-', color=info['color'], markersize=6, linewidth=2)
    ax.set_xlabel('K', fontsize=10)
    ax.set_ylabel(metric, fontsize=10)
    ax.set_title(info['title'], fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3)

    if info['lower_better']:
        best_idx = metrics_df[metric].idxmin()
    else:
        best_idx = metrics_df[metric].idxmax()
    best_k = metrics_df.loc[best_idx, 'K']
    best_v = metrics_df.loc[best_idx, metric]
    ax.axvline(x=best_k, color=info['color'], linestyle='--', alpha=0.4)
    ax.annotate(f'best K={best_k}\n({best_v:.4f})',
                xy=(best_k, best_v), fontsize=8, color=info['color'],
                textcoords="offset points", xytext=(10, 5),
                arrowprops=dict(arrowstyle='->', color=info['color'], lw=0.8))

    for ek in EXPORT_KS:
        row_ek = metrics_df[metrics_df['K'] == ek]
        if len(row_ek) > 0:
            ax.plot(ek, row_ek.iloc[0][metric], 's',
                    color=info['color'], markersize=10,
                    markerfacecolor='none', markeredgewidth=2)

axes_flat[-1].set_visible(False)
fig_met.suptitle(f"K-Means Quality Metrics (seed={SEED}, K={K_MIN}..{K_MAX})\n"
                 f"Squares mark exported K values: {EXPORT_KS}",
                 fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# ═══════════════════════════════════════════════════════════════
#  Part 2: 1D Sankey (full range)
# ═══════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("1D Cluster Embedding with Sankey Flow (full range)")
print("=" * 70)
plot_1d_sankey((K_MIN, K_MAX), 'MDS', True, True, 0.20)

# ═══════════════════════════════════════════════════════════════
#  Part 3: Maps + Centroid Embedding + Profiles for each K
# ═══════════════════════════════════════════════════════════════

# Pre-merge (reuse if already done)
geo_exp = nuts3_shp[[SHP_ID_COL, 'geometry']].copy()
geo_exp = geo_exp.rename(columns={SHP_ID_COL: 'ID_NUTS3'})
geo_exp['ID_NUTS3'] = geo_exp['ID_NUTS3'].astype(str)

geo_merged_exp = geo_exp.merge(
    df[['ID_NUTS3', 'Name', 'ID_NUTS0', 'total_population'] + FEATURE_COLS],
    on='ID_NUTS3', how='left')

for ki, k in enumerate(EXPORT_KS):
    print(f"\n{'=' * 70}")
    print(f"K = {k}  ({ki+1}/{len(EXPORT_KS)})")
    print(f"{'=' * 70}")

    labels = cluster_assignments[k]
    cd = centroid_data[k]
    colors = cd['colors']

    # Prepare geodataframe
    label_df = pd.DataFrame({'ID_NUTS3': df['ID_NUTS3'], 'Cluster': labels})
    gdf = geo_merged_exp.copy()
    gdf = gdf.merge(label_df, on='ID_NUTS3', how='left')
    gdf['Cluster'] = gdf['Cluster'].fillna(-1).astype(int)

    # ── Figure: Map + Centroid Embedding ──
    fig = plt.figure(figsize=(16, 14))
    gs = GridSpec(2, 2, figure=fig,
                  height_ratios=[2.5, 2],
                  width_ratios=[2, 1],
                  hspace=0.15, wspace=0.05)

    ax_map = fig.add_subplot(gs[0, :])       # map spans full width top
    ax_embed = fig.add_subplot(gs[1, 0])     # centroid embed bottom-left
    ax_profile = fig.add_subplot(gs[1, 1])   # line profile bottom-right

    # ── Map ──
    unmatched = gdf[gdf['Cluster'] == -1]
    if len(unmatched) > 0:
        unmatched.plot(ax=ax_map, color='#E0E0E0',
                       edgecolor='#CCCCCC', linewidth=0.3)

    matched = gdf[gdf['Cluster'] >= 0].copy()
    for c_idx in range(k):
        subset = matched[matched['Cluster'] == c_idx]
        if len(subset) > 0:
            subset.plot(ax=ax_map, color=colors[c_idx],
                        edgecolor='#999999', linewidth=0.3)

    legend_patches = [
        Patch(facecolor=colors[c], edgecolor='black',
              label=f'{c} (n={(labels==c).sum()})')
        for c in range(k)]
    ax_map.legend(handles=legend_patches, loc='lower left',
                  fontsize=6, framealpha=0.9, title='Cluster',
                  title_fontsize=8, ncol=2 if k > 15 else 1)

    nuts0_shp.boundary.plot(ax=ax_map, color='black', linewidth=1.0)

    row = metrics_df[metrics_df['K'] == k].iloc[0]
    info_text = (f"Silhouette: {row['Silhouette']:.4f}\n"
                 f"Var.Explained: {row['Variance_Explained']:.1f}%\n"
                 f"Davies-Bouldin: {row['Davies_Bouldin']:.4f}")
    ax_map.text(0.98, 0.98, info_text, transform=ax_map.transAxes,
                fontsize=9, verticalalignment='top',
                horizontalalignment='right',
                bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

    ax_map.set_title(
        f"NUTS3 Population Clusters — K={k}, seed={SEED}, data years {YEAR_MIN}–{YEAR_MAX}",
        fontsize=13, fontweight='bold')
    ax_map.set_axis_off()
    ax_map.set_xlim(-25, 45)
    ax_map.set_ylim(34, 72)

    # ── Centroid 2D embedding ──
    coords = cd['coords_2d']
    sizes = [(labels == c).sum() for c in range(k)]
    max_size = max(sizes) if max(sizes) > 0 else 1
    marker_sizes = [max(100, 600 * s / max_size) for s in sizes]

    for c in range(k):
        ax_embed.scatter(coords[c, 0], coords[c, 1],
                         c=colors[c], s=marker_sizes[c],
                         edgecolors='black', linewidths=0.8, zorder=3)
        ax_embed.annotate(f'{c}', (coords[c, 0], coords[c, 1]),
                          ha='center', va='center',
                          fontsize=8, fontweight='bold')

    dist_matrix = sqf(pdist(cd['centroids']))
    for i in range(k):
        nearest = np.argsort(dist_matrix[i])[1]
        ax_embed.plot(
            [coords[i, 0], coords[nearest, 0]],
            [coords[i, 1], coords[nearest, 1]],
            color='gray', alpha=0.3, linewidth=0.8, zorder=1)

    ax_embed.set_title('Centroid 2D embedding\n(size ∝ population)',
                       fontsize=10, fontweight='bold')
    ax_embed.set_xlim(-0.1, 1.1)
    ax_embed.set_ylim(-0.1, 1.1)
    ax_embed.set_aspect('equal')
    ax_embed.grid(True, alpha=0.2)

    # ── Z-scored line profile ──
    centroids_z = cd['centroids']
    centroids_raw = np.array([
        df.loc[labels == c, FEATURE_COLS].mean().values
        for c in range(k)])

    x_pos = np.arange(len(FEATURE_COLS))
    ax_profile.axhline(y=0, color='black', linewidth=1.5,
                       linestyle='-', alpha=0.3)

    for c in range(k):
        n_pts = (labels == c).sum()
        ax_profile.plot(x_pos, centroids_z[c], 'o-',
                        color=colors[c], linewidth=2, markersize=6,
                        markeredgecolor='black', markeredgewidth=0.4,
                        label=f'{c} ({n_pts})', alpha=0.85)

    ax_profile.set_xticks(x_pos)
    ax_profile.set_xticklabels(
        [f.replace('pct_', '') for f in FEATURE_COLS],
        rotation=40, ha='right', fontsize=8)
    ax_profile.set_ylabel('Z-score', fontsize=9)
    ax_profile.set_title('Centroid profiles (Z-scored)',
                         fontsize=10, fontweight='bold')
    ax_profile.grid(True, axis='y', alpha=0.3)
    ax_profile.legend(fontsize=5, loc='best', framealpha=0.8,
                      ncol=2 if k > 12 else 1, title='Cluster',
                      title_fontsize=6)

    plt.tight_layout()
    plt.show()

    # ── Tables ──
    profile_z = pd.DataFrame(centroids_z, columns=FEATURE_COLS)
    profile_z.index.name = 'Cluster'
    profile_z['N'] = [(labels == c).sum() for c in range(k)]
    cols_z = ['N'] + FEATURE_COLS
    profile_z = profile_z[cols_z]

    def z_color_cells(val):
        if isinstance(val, (int, np.integer)):
            return ''
        if val > 0.5:
            return f'background-color: rgba(0,128,0,{min(abs(val)/3, 0.4):.2f})'
        elif val < -0.5:
            return f'background-color: rgba(200,0,0,{min(abs(val)/3, 0.4):.2f})'
        return ''

    def row_border(row):
        c_idx = int(row.name)
        col = colors[c_idx] if c_idx < len(colors) else '#999999'
        return [f'border-left: 5px solid {col}'] + [''] * (len(row) - 1)

    styled_z = (profile_z.style
                .map(z_color_cells)
                .apply(row_border, axis=1)
                .format(precision=2))
    display(HTML(f"<h4>Z-scored Centroids (K={k})</h4>"))
    display(styled_z)

    profile = cluster_profile(k, labels)

    def color_rows(row):
        c_idx = int(row.name)
        bg = colors[c_idx] if c_idx < len(colors) else '#FFFFFF'
        r, g, b = int(bg[1:3],16)/255, int(bg[3:5],16)/255, int(bg[5:7],16)/255
        lum = 0.299*r + 0.587*g + 0.114*b
        fg = '#000000' if lum > 0.45 else '#FFFFFF'
        return [f'background-color: {bg}; color: {fg}'] * len(row)

    styled_raw = (profile.style
                  .apply(color_rows, axis=1)
                  .format(precision=2))
    display(HTML(f"<h4>Cluster Profiles — raw values (K={k})</h4>"))
    display(styled_raw)

# ═══════════════════════════════════════════════════════════════
#  Part 4: TRANSITION ANALYSIS — Static Examples
# ═══════════════════════════════════════════════════════════════

print("\n" + "═" * 70)
print("  PART 4: CLUSTER TRANSITION ANALYSIS")
print("  (Auto-selected representative examples for static export)")
print("═" * 70)

# ── Select K pairs for transition analysis ──
# Use two exported K values that are reasonably spaced apart
export_ks_sorted = sorted(EXPORT_KS)
if len(export_ks_sorted) >= 2:
    # Pick K₁ from lower half, K₂ from upper half
    mid = len(export_ks_sorted) // 2
    TRANS_K1 = export_ks_sorted[max(0, mid - 1)]
    TRANS_K2 = export_ks_sorted[min(len(export_ks_sorted) - 1, mid + 1)]
    # Ensure they're different
    if TRANS_K1 == TRANS_K2:
        TRANS_K1 = export_ks_sorted[0]
        TRANS_K2 = export_ks_sorted[-1]
else:
    TRANS_K1 = K_MIN
    TRANS_K2 = K_MAX

print(f"\n  Transition pair: K₁={TRANS_K1} → K₂={TRANS_K2}")

# ─────────────────────────────────────────────────────────────
#  4a: Transition FROM — largest cluster at K₁ → K₂
# ─────────────────────────────────────────────────────────────

labels_k1 = cluster_assignments[TRANS_K1]
cluster_sizes_k1 = [(labels_k1 == c).sum() for c in range(TRANS_K1)]
largest_c1 = int(np.argmax(cluster_sizes_k1))

print(f"\n{'─' * 70}")
print(f"  4a: Transition FROM K={TRANS_K1}.cluster{largest_c1} "
      f"(largest, n={cluster_sizes_k1[largest_c1]}) → K={TRANS_K2}")
print(f"{'─' * 70}")

display(HTML(f"<h3>🔀 Transition FROM: K={TRANS_K1}.cluster{largest_c1} → K={TRANS_K2}</h3>"
             f"<p><em>Tracking where the largest cluster at K={TRANS_K1} disperses "
             f"when granularity increases to K={TRANS_K2}</em></p>"))

show_transitions_from_cluster(TRANS_K1, largest_c1, TRANS_K2)

# ─────────────────────────────────────────────────────────────
#  4b: Transition TO — largest cluster at K₂ ← K₁
# ─────────────────────────────────────────────────────────────

labels_k2 = cluster_assignments[TRANS_K2]
cluster_sizes_k2 = [(labels_k2 == c).sum() for c in range(TRANS_K2)]
largest_c2 = int(np.argmax(cluster_sizes_k2))

print(f"\n{'─' * 70}")
print(f"  4b: Transition TO K={TRANS_K2}.cluster{largest_c2} "
      f"(largest, n={cluster_sizes_k2[largest_c2]}) ← K={TRANS_K1}")
print(f"{'─' * 70}")

display(HTML(f"<h3>🔀 Transition TO: K={TRANS_K1} → K={TRANS_K2}.cluster{largest_c2}</h3>"
             f"<p><em>Tracing origins: where do members of the largest cluster at "
             f"K={TRANS_K2} come from at K={TRANS_K1}?</em></p>"))

show_transitions_to_cluster(TRANS_K1, TRANS_K2, largest_c2)

# ─────────────────────────────────────────────────────────────
#  4c: Pair Transition — find best Jaccard match between K₁ & K₂
# ─────────────────────────────────────────────────────────────

print(f"\n{'─' * 70}")
print(f"  4c: Finding best-matching cluster pair (highest Jaccard)...")
print(f"{'─' * 70}")

best_jaccard = -1
best_pair = (0, 0)

for c1 in range(TRANS_K1):
    mask_c1 = (labels_k1 == c1)
    n_c1 = int(mask_c1.sum())
    if n_c1 == 0:
        continue
    for c2 in range(TRANS_K2):
        mask_c2 = (labels_k2 == c2)
        n_c2 = int(mask_c2.sum())
        if n_c2 == 0:
            continue
        n_shared = int((mask_c1 & mask_c2).sum())
        jaccard = n_shared / (n_c1 + n_c2 - n_shared) if (n_c1 + n_c2 - n_shared) > 0 else 0
        if jaccard > best_jaccard:
            best_jaccard = jaccard
            best_pair = (c1, c2)

pair_c1, pair_c2 = best_pair
print(f"  Best pair: K={TRANS_K1}.C{pair_c1} ↔ K={TRANS_K2}.C{pair_c2} "
      f"(Jaccard={best_jaccard:.3f})")

display(HTML(f"<h3>🔀 Pair Transition: K={TRANS_K1}.C{pair_c1} → "
             f"K={TRANS_K2}.C{pair_c2}  (Jaccard={best_jaccard:.3f})</h3>"
             f"<p><em>Three-class decomposition of the strongest cluster "
             f"continuity between K={TRANS_K1} and K={TRANS_K2}</em></p>"))

show_transition_between(TRANS_K1, pair_c1, TRANS_K2, pair_c2)

# ─────────────────────────────────────────────────────────────
#  4d: Additional pair — lowest Jaccard (most disrupted) for contrast
# ─────────────────────────────────────────────────────────────

# Find a cluster at K1 that splits most evenly (lowest max-destination share)
print(f"\n{'─' * 70}")
print(f"  4d: Finding most-split cluster at K={TRANS_K1} → K={TRANS_K2}...")
print(f"{'─' * 70}")

most_split_c1 = 0
lowest_max_share = 1.0

for c1 in range(TRANS_K1):
    mask_c1 = (labels_k1 == c1)
    n_c1 = int(mask_c1.sum())
    if n_c1 < 20:  # skip very small clusters
        continue
    dest_labels = labels_k2[mask_c1]
    _, counts = np.unique(dest_labels, return_counts=True)
    max_share = counts.max() / n_c1
    if max_share < lowest_max_share:
        lowest_max_share = max_share
        most_split_c1 = c1

# Only show if it's different from the previous examples
if most_split_c1 != largest_c1:
    print(f"  Most fragmented: K={TRANS_K1}.C{most_split_c1} "
          f"(largest destination = {lowest_max_share*100:.1f}%)")

    display(HTML(f"<h3>🔀 Most-Split Cluster: K={TRANS_K1}.C{most_split_c1} → "
                 f"K={TRANS_K2}</h3>"
                 f"<p><em>The cluster that fragments most when K increases — "
                 f"largest destination receives only {lowest_max_share*100:.1f}% "
                 f"of members</em></p>"))

    show_transitions_from_cluster(TRANS_K1, most_split_c1, TRANS_K2)
else:
    print(f"  (Skipped — same as 4a)")

# ═══════════════════════════════════════════════════════════════
#  Summary
# ═══════════════════════════════════════════════════════════════

print(f"\n{'═' * 70}")
print(f"  STATIC EXPORT COMPLETE")
print(f"{'═' * 70}")
print(f"  Included:")
print(f"    • Quality metrics (5 charts)")
print(f"    • 1D Sankey flow (K={K_MIN}..{K_MAX})")
print(f"    • Maps + profiles for K = {EXPORT_KS}")
print(f"    • Transition FROM: K={TRANS_K1}.C{largest_c1} → K={TRANS_K2}")
print(f"    • Transition TO: K={TRANS_K1} → K={TRANS_K2}.C{largest_c2}")
print(f"    • Pair transition: K={TRANS_K1}.C{pair_c1} ↔ K={TRANS_K2}.C{pair_c2} "
      f"(Jaccard={best_jaccard:.3f})")
if most_split_c1 != largest_c1:
    print(f"    • Most-split: K={TRANS_K1}.C{most_split_c1} → K={TRANS_K2} "
          f"(max dest. share={lowest_max_share*100:.1f}%)")
print(f"\n  To export:")
print(f"    File → Download as → HTML (.html)")
print(f"    or: jupyter nbconvert --to html --execute kMeans-EU-NUTS3.ipynb")
print(f"{'═' * 70}")

## 📤 How to Export This Notebook to HTML

### Option A: Static export (recommended for sharing)

1. **Run the "Static Export" cell above** — it generates all key visualizations

   as pure matplotlib figures and styled tables
2. **File → Download as → HTML (.html)**

All static plots, tables, and the Sankey diagram will be included.
Interactive slider-based cells will show their **last rendered state**.

### Option B: From command line (with execution)

```bash
jupyter nbconvert --to html --execute --ExecutePreprocessor.timeout=300 kMeans-EU-NUTS3.ipynb